# T4D Full Project Lite

This notebook keeps only the path that feeds the trained models and the forward projections.

What stays:
- baseline station signatures and regime clustering
- daily feature engineering
- station and station-year cleanup used by the training split
- phase 6 air-to-water temperature modeling
- phase 7 water-property response modeling
- phase 9 forward projections, scan summaries, and model bundle export

What drops:
- most exploratory charts
- lag diagnostics that do not feed the model inputs directly
- optional benchmark and asset-manifest side quests

The goal here is not to be clever. The goal is to be linear, readable, and runnable end to end.


## Setup

Intent:
- load the cleaned water history
- attach lightweight station labels
- keep only the helper bits that later modeling and forecasting cells actually use

Result to expect:
- one normalized `water` table
- one `station_lookup` table
- a few tiny lookup helpers for readable joins and labels


In [ ]:
# first... dependencies
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [ ]:
# quick helper to attach station context to any frame with region and station columns, without worrying about duplicates or column conflicts

def attach_station_context( frame ):
    if not { "region", "station" }.issubset( frame.columns ):
        return frame

    overlap_cols = [ 
        col
        for col in station_context_cols
        if col in frame.columns and col not in [ "region", "station" ]
    ]
    base = frame.drop( columns = overlap_cols, errors = "ignore" )
    return base.merge( station_lookup[ station_context_cols ], on = [ "region", "station" ], how = "left" )

In [ ]:
# let's get some station context in the row labels for readability
def station_row_label( region, station ):
    region_key = str( region ).strip( ).lower( )
    station_key = str( station ).strip( ).lower( )[ -2: ]
    station_name = station_name_lookup.get( ( region_key, station_key ) )
    region_name = region_name_lookup.get( region_key, region_key )

    if pd.isna( station_name ) or station_name is None or str( station_name ).strip( ) == "":
        return f"({ region_key }{ station_key }) { region_name }"

    return f"({ region_key }{ station_key }) { station_name}, { region_name }"

In [ ]:
# and then we can use that to add row labels to any frame with region and station columns, 
# without worrying about duplicates or column conflicts
def add_station_row_labels( frame, label_col = "row_label" ):
    labeled = attach_station_context( frame )
    labeled[ label_col ] = labeled.apply( 
        lambda row: station_row_label( row[ "region" ], row[ "station" ] ),
        axis = 1,
    )
    return labeled

In [ ]:
# keep charts readable, even though this notebook is mostly tables and models
sns.set_theme( style = "whitegrid" )

# and now, let's use the notebook's expected relative layout directly
# note we also had 1min data from NERRS, 1hr from ERA5
# but online colabs like google couldn't handle it even when we reduced data to 4hr resolution
# so while 1hr data is the most honest, gotta get it from onedrive
res = 1

data_dir = Path( "../data" ) 
ref_dir = data_dir / "reference" # re: station lookup, and also the 1hr water history data that we couldn't get from onedrive
artifact_dir = Path( "../artifacts" )
artifact_dir.mkdir( parents = True, exist_ok = True ) # get created later

#sys.path.append( str( Path( ".." ).resolve( ) ) )

water = pd.read_csv( data_dir / f"{ res }hr" / f"t4d.{ res }hr.water.history.csv" )
# lil easier to read
water = water.rename( 
    columns = { 
        "w_temp_c": "water_temp",            # celsius
        "w_sal_psu": "salinity",             # practical salinity units (psu)
        "w_do_mg_l": "oxygen",               # milligrams per liter (mg/L)
        "w_do_pct": "oxy_saturation",        # percent saturation (%)
        "depth_m": "depth",                  # meters (m)
        "w_ph": "ph",                        # pH units ... heh
        "m_wind_ms": "wind_speed",           # meters per second (m/s) (math'd from u10 and v10)
        "m_ssrd_kwh_m2": "solar_radiation",  # kilowatt-hours per square meter (kWh/m^2) (from ssrd)
        "m_precip_mmh": "precipitation",     # millimeters per hour (mm/h) (adj from tp)
        "m_temp_c": "air_temp",              # celsius (from kelvin in ERA5)
    }
)
water[ "region" ] = water[ "region" ].astype( str ).str.strip( ).str.lower( )
water[ "station" ] = water[ "station" ].astype( str ).str.strip( ).str.lower( )

# and now, let's drop the globally out-of-scope region
water = water.loc[ water[ "region" ] != "hee" ]

station_lookup = pd.read_csv( ref_dir / "nerrs_station_index.csv" )
station_lookup[ "region" ] = station_lookup[ "region_code" ].astype( str ).str.strip( ).str.lower( )
station_lookup[ "station_full" ] = station_lookup[ "station" ].astype( str ).str.strip( ).str.lower( )
station_lookup[ "station" ] = station_lookup[ "station_full" ].str[ -2: ]
station_lookup = station_lookup[ 
    [ 
        "region",
        "station",
        "station_full",
        "region_name",
        "station_name",
        "latitude",
        "longitude",
        "in_t4d_1hr_water_history",
    ]
].drop_duplicates( subset = [ "region", "station" ] )

# we can basically 'enum' these in (like a factor in R)
# so not that wasteful to have around for readability
region_name_lookup = ( 
    station_lookup
    .dropna( subset = [ "region_name" ] )
    .drop_duplicates( subset = [ "region" ] )
    .set_index( "region" )[ "region_name" ]
    .to_dict( )
)

station_name_lookup = ( 
    station_lookup
    .dropna( subset = [ "station_name" ] )
    .set_index( [ "region", "station" ] )[ "station_name" ]
    .to_dict( )
)

station_context_cols = [ 
    "region",
    "station",
    "station_full",
    "region_name",
    "station_name",
    "latitude",
    "longitude",
    "in_t4d_1hr_water_history",
]

water = attach_station_context( water )

print( f"water rows: { len( water ) }" )
print( f"station lookup rows: { len( station_lookup ) }" )


In [ ]:
# just the facts, doc
water = water.drop( columns = [ "region", "station" ] ) # redundant
water[ "region_name" ] = water[ "region_name" ].astype( "category" ) # more efficient
water[ "station_name" ] = water[ "station_name" ].astype( "category" ) # same
water[ "station_full" ] = water[ "station_full" ].astype( "category" ) # also redundant, but might as well
water[ "datetime" ] = pd.to_datetime( water[ "datetime" ] ) # make sure it's a datetime
#water.describe().T
water.info()

## Regime Context

Intent:
- build a station-level baseline from the early valid years at each station
- turn that baseline into a small, interpretable regime label used later by phase 5 and phase 9

Result to expect:
- `station_baseline` and `station_baseline_display`
- `cluster_station` as the station-level regime table
- clustering metadata saved in memory for the final model bundle


In [ ]:
# station character baseline (first valid years, not fixed 1995-2000)
# this keeps the idea simple: each station gets its own early baseline window

# leave water intact for now...
base = water.copy( )
base = base.loc[ 
    :, [ 
        'station_full', # lookup code
        'region_name',
        'station_name',
        'datetime',
        'water_temp',
        'salinity',
        'oxygen',
        'oxy_saturation',
        'ph',
        'depth'
    ]
].dropna( subset = [ 'datetime' ] )
base[ 'year' ] = base[ 'datetime' ].dt.year
#base.describe( ).T

In [ ]:
# annual coverage check so thin years do not define the baseline
year_obs = ( 
    base
    .groupby( [ 'region_name', 'station_name', 'year' ], as_index = False )
    .agg( n_obs = ( 'datetime', 'size' ) )
)

# leap years and all that jazz
# prolly not a big diff, though
year_obs[ 'expected_obs' ] = np.where( 
    year_obs[ 'year' ] % 4 == 0,
    366 * 24,
    365 * 24,
)

#year_obs.describe( ).T

In [ ]:
# try to identify years with enough coverage to be considered valid for defining the baseline at all
year_obs[ 'coverage_frac' ] = year_obs[ 'n_obs' ] / year_obs[ 'expected_obs' ]
year_obs[ 'year_is_valid' ] = year_obs[ 'coverage_frac' ] >= 0.70 

#year_obs.describe( ).T
#year_obs[ 'year_is_valid' ].value_counts( )

In [ ]:
valid_years = year_obs.loc[ year_obs[ 'year_is_valid' ] ].copy( )
valid_years = valid_years.sort_values( [ 'region_name', 'station_name', 'year' ] ).reset_index( drop = True )
valid_years[ 'valid_year_rank' ] = valid_years.groupby( [ 'region_name', 'station_name' ] ).cumcount( ) + 1
#valid_years.sample( 10 )
del( year_obs ) # done with it

In [ ]:
# turned out too many didn't even start operating between 95 and 00
# so had to rework to find their first five instead
baseline_years = valid_years.loc[ valid_years[ 'valid_year_rank' ] <= 5 ].copy( )

baseline_window = ( 
    baseline_years
    .groupby( [ 'region_name', 'station_name' ], as_index = False )
    .agg( 
        baseline_start_year = ( 'year', 'min' ),
        baseline_end_year = ( 'year', 'max' ),
        n_valid_years = ( 'year', 'size' ),
        mean_year_coverage = ( 'coverage_frac', 'mean' ),
    )
)

# is there enough?
baseline_window[ 'insufficient_baseline' ] = baseline_window[ 'n_valid_years' ] < 5
#baseline_window.describe( ).T
#baseline_window.sample( 10 )

In [ ]:
# now we can merge that baseline window info back to the base data, 
# and flag which rows are in the baseline window for each station
# so here, a func
def build_season_signature( group ):
    group = group.sort_values( 'month' ).copy( )

    temp_series = group[ 'water_temp_mmean' ]
    oxygen_series = group[ 'oxygen_mmean' ]

    # if there are any valid temp values, we can define warm and cold months based on the mean
    if temp_series.notna( ).any( ):
        temp_mean = float( temp_series.mean( ) )
        warm_mask = ( temp_series > temp_mean ).fillna( False )
        cold_mask = ( temp_series < temp_mean ).fillna( False )

        if not warm_mask.any( ):
            warm_mask = ( temp_series == temp_series.max( ) ).fillna( False )

        if not cold_mask.any( ):
            cold_mask = ( temp_series == temp_series.min( ) ).fillna( False )

        warm_peak_month = int( group.loc[ temp_series.idxmax( ), 'month' ] )

    # if there are no valid temp values, we can't define warm and cold months,
    # ... so set everything to NaN or False
    else:
        temp_mean = np.nan
        warm_mask = pd.Series( False, index = group.index )
        cold_mask = pd.Series( False, index = group.index )
        warm_peak_month = np.nan

    warm_oxygen = group.loc[ warm_mask, 'oxygen_mmean' ].dropna( )
    cold_oxygen = group.loc[ cold_mask, 'oxygen_mmean' ].dropna( )

    # if there are any valid oxygen values, we can find the lowest/worst month?
    if oxygen_series.notna( ).any( ):
        oxygen_nadir_month = int( group.loc[ oxygen_series.idxmin( ), 'month' ] )

    else:
        oxygen_nadir_month = np.nan

    # if we have both a warm peak month and an oxygen crash, we can calculate the phase lag in months
    if pd.notna( warm_peak_month ) and pd.notna( oxygen_nadir_month ):
        temp_oxygen_phase_lag = float( ( oxygen_nadir_month - warm_peak_month ) % 12 )

    else:
        temp_oxygen_phase_lag = np.nan

    # and then we can return all the signature stats for this station as a series
    # which will become columns in the final frame
    return pd.Series( { 
        'warm_peak_month': warm_peak_month,
        'oxygen_nadir_month': oxygen_nadir_month,
        'temp_oxygen_phase_lag': temp_oxygen_phase_lag,
        'warm_len_months': int( warm_mask.sum( ) ),
        'cold_len_months': int( cold_mask.sum( ) ),
        'warm_intensity_temp': float( group.loc[ warm_mask, 'water_temp_mmean' ].sub( temp_mean ).sum( ) ) if pd.notna( temp_mean ) and warm_mask.any( ) else np.nan,
        'cold_intensity_temp': float( temp_mean * int( cold_mask.sum( ) ) - group.loc[ cold_mask, 'water_temp_mmean' ].sum( ) ) if cold_mask.any( ) else 0.0,
        'warm_oxygen_mean': float( warm_oxygen.mean( ) ) if len( warm_oxygen ) > 0 else np.nan,
        'warm_oxygen_min': float( warm_oxygen.min( ) ) if len( warm_oxygen ) > 0 else np.nan,
        'cold_oxygen_mean': float( cold_oxygen.mean( ) ) if len( cold_oxygen ) > 0 else np.nan,
    } )

In [ ]:
# keep stations with at least 3 valid years
eligible_stations = baseline_window.loc[ 
    baseline_window[ 'n_valid_years' ] >= 5,
    [ 'region_name', 'station_name' ],
]

# pull only rows from each station's selected baseline years
base = base.merge( 
    baseline_years[ [ 'region_name', 'station_name', 'year' ] ],
    on = [ 'region_name', 'station_name', 'year' ],
    how = 'inner',
)

# then keep only stations that passed the >=5-year minimum
base = base.merge( eligible_stations, on = [ 'region_name', 'station_name' ], how = 'inner' )
#eligible_stations.sample(5)
#eligible_stations.describe( ).T
#base.sample( 10 )

In [ ]:
# okay, now build station character features from the selected baseline window
base[ 'month' ] = base[ 'datetime' ].dt.month
# get mean of each year per station, then the mean of those five years per station
# yearly means first ( equal-year weighting )
annual = ( 
    base
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( 
        water_temp_ymean = ( 'water_temp', 'mean' ),
        salinity_ymean = ( 'salinity', 'mean' ),
        oxygen_ymean = ( 'oxygen', 'mean' ),
        saturation_ymean = ( 'oxy_saturation', 'mean' ),
        ph_ymean = ( 'ph', 'mean' ),
        depth_ymean = ( 'depth', 'mean' ),
    )
)

# each station gets its own, yo
annual_means = ( 
    annual
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        mean_annual_water_temp = ( 'water_temp_ymean', 'mean' ),
        mean_annual_salinity = ( 'salinity_ymean', 'mean' ),
        mean_annual_oxygen = ( 'oxygen_ymean', 'mean' ),
        mean_annual_saturation = ( 'saturation_ymean', 'mean' ),
        mean_annual_ph = ( 'ph_ymean', 'mean' ),
        mean_annual_depth = ( 'depth_ymean', 'mean' ),
        n_years_water_temp = ( 'water_temp_ymean', 'count' ),
        n_years_salinity = ( 'salinity_ymean', 'count' ),
        n_years_oxygen = ( 'oxygen_ymean', 'count' ),
        n_years_saturation = ( 'saturation_ymean', 'count' ),
        n_years_ph = ( 'ph_ymean', 'count' ),
        n_years_depth = ( 'depth_ymean', 'count' ),
    )
)

# seasonal amplitudes from monthly climatology
monthly = ( 
    base
    .groupby( [ 'region', 'station', 'month' ], as_index = False )
    .agg( 
        water_temp_mmean = ( 'water_temp', 'mean' ),
        salinity_mmean = ( 'salinity', 'mean' ),
        oxygen_mmean = ( 'oxygen', 'mean' ),
        saturation_mmean = ( 'oxy_saturation', 'mean' ),
        ph_mmean = ( 'ph', 'mean' ),
        depth_mmean = ( 'depth', 'mean' ),
    )
)

# build both a simple amplitude table and a richer warm / cold season signature table
seasonal_amp = ( 
    monthly
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        seasonal_amp_water_temp = ( 'water_temp_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_salinity = ( 'salinity_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_oxygen = ( 'oxygen_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_saturation = ( 'saturation_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_ph = ( 'ph_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_depth = ( 'depth_mmean', lambda s: s.max( ) - s.min( ) ),
    )
)

seasonal_signature = ( 
    monthly
    .groupby( [ 'region', 'station' ] )
    .apply( build_season_signature )
    .reset_index( )
)

# depth summary ( useful if sensor environment differs by station )
# let's see if the recorders are variable depth per station, or predictable?
depth_stats = ( 
    base
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        median_depth = ( 'depth', 'median' ),
        p10_depth = ( 'depth', lambda s: s.quantile( 0.10 ) ),
        p90_depth = ( 'depth', lambda s: s.quantile( 0.90 ) ),
    )
)
depth_stats[ 'iqr_depth' ] = depth_stats[ 'p90_depth' ] - depth_stats[ 'p10_depth' ]

coverage = ( 
    base
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        n_obs_total = ( 'datetime', 'size' ),
        n_years_present = ( 'year', 'nunique' ),
    )
)

# lets snap these together into a baseline ...
station_baseline = ( 
    annual_means
    .merge( seasonal_amp, on = [ 'region', 'station' ], how = 'left' )
    .merge( seasonal_signature, on = [ 'region', 'station' ], how = 'left' )
    .merge( depth_stats, on = [ 'region', 'station' ], how = 'left' )
    .merge( coverage, on = [ 'region', 'station' ], how = 'left' )
    .merge( 
        baseline_window[ [ 
            'region',
            'station',
            'baseline_start_year',
            'baseline_end_year',
            'n_valid_years',
            'mean_year_coverage',
            'is_partial_baseline',
        ] ],
        on = [ 'region', 'station' ],
        how = 'left',
    )
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

# placeholder flag in case we later add nearest-neighbor feature fallback
station_baseline[ 'character_imputed' ] = False
station_baseline_display = ( 
    station_baseline
    .merge( 
        station_lookup[ [ 'region', 'station', 'region_name', 'station_name', 'latitude', 'longitude' ] ],
        on = [ 'region', 'station' ],
        how = 'left',
    )
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)



# and now, let's attach readable station labels once
station_baseline_display = attach_station_context( station_baseline )

print( f"stations with >=3 valid baseline years: { eligible_stations.shape[ 0 ] }" )
print( f"station baseline rows: { len( station_baseline ) }" )


In [ ]:
# single, slim KMeans pass with a domain-driven feature subset
# selected features:
# - broad baseline hydroclimate structure only
# - keep the richer warm / cold season signature fields in station_baseline,
#   but do not let them define the primary regime geometry
domain_feature_cols = [ 
    'mean_annual_water_temp',
    'seasonal_amp_water_temp',
    'mean_annual_salinity',
    'seasonal_amp_salinity',
    'mean_annual_oxygen',
    'seasonal_amp_oxygen',
    'mean_annual_depth',
]

print( 'domain features:', domain_feature_cols )

domain_input = station_baseline[ [ 'region', 'station' ] + domain_feature_cols ].copy( )

# simple missing-value strategy: region median, then global median
for col in domain_feature_cols:
    domain_input[ col ] = domain_input.groupby( 'region' )[ col ].transform( lambda s: s.fillna( s.median( ) ) )
    domain_input[ col ] = domain_input[ col ].fillna( domain_input[ col ].median( ) )

scaler_domain = StandardScaler( )
X_scaled_domain = scaler_domain.fit_transform( domain_input[ domain_feature_cols ] )

# k scan for elbow + silhouette
k_scan_rows = [ ]

for k in range( 2, 11 ):
    km_scan = KMeans( n_clusters = k, random_state = 42, n_init = 20 )
    labels_scan = km_scan.fit_predict( X_scaled_domain )

    k_scan_rows.append( { 
        'k': k,
        'inertia': float( km_scan.inertia_ ),
        'silhouette': float( silhouette_score( X_scaled_domain, labels_scan ) ),
    } )

k_scan_domain = pd.DataFrame( k_scan_rows )
k_scan_domain[ 'inertia_drop' ] = k_scan_domain[ 'inertia' ].shift( 1 ) - k_scan_domain[ 'inertia' ]
k_scan_domain[ 'inertia_drop_pct' ] = k_scan_domain[ 'inertia_drop' ] / k_scan_domain[ 'inertia' ].shift( 1 )

recommended_k = int( k_scan_domain.loc[ k_scan_domain[ 'silhouette' ].idxmax( ), 'k' ] )

# keep the final pick interpretable for section 1.2
k_min_interpretable = 3
k_max_interpretable = 6
k_scan_interpretable = k_scan_domain.loc[ k_scan_domain[ 'k' ].between( k_min_interpretable, k_max_interpretable ) ].copy( )

if k_scan_interpretable.empty:
    recommended_k_interpretable = 4

else:
    recommended_k_interpretable = int( 
        k_scan_interpretable.loc[ k_scan_interpretable[ 'silhouette' ].idxmax( ), 'k' ]
    )

print( '\ndomain-feature k scan:' )
print( k_scan_domain.round( 4 ) )
print( f'unbounded silhouette max K: { recommended_k }' )
print( f'interpretable-window K ( { k_min_interpretable }-{ k_max_interpretable } ): { recommended_k_interpretable }' )

fig, axes = plt.subplots( 1, 2, figsize = ( 14, 5 ) )

axes[ 0 ].plot( 
    k_scan_domain[ 'k' ],
    k_scan_domain[ 'inertia' ],
    marker = 'o'
)
axes[ 0 ].set_title( 'Domain Features: K vs Inertia' )
axes[ 0 ].set_xlabel( 'K' )
axes[ 0 ].set_ylabel( 'Inertia' )

axes[ 1 ].plot( 
    k_scan_domain[ 'k' ],
    k_scan_domain[ 'silhouette' ],
    marker = 'o'
)
axes[ 1 ].set_title( 'Domain Features: K vs Silhouette Score' )
axes[ 1 ].set_xlabel( 'K' )
axes[ 1 ].set_ylabel( 'Silhouette Score' )

plt.tight_layout( )
plt.show( )

# not a really DISTINCT elbow, but a clear flattening after 4 or 5 clusters, and a silhouette max at 5

# use the bounded recommendation by default; override manually if needed
k_clusters = recommended_k_interpretable
kmeans_model = KMeans( n_clusters = k_clusters, random_state = 42, n_init = 20 )
domain_input[ 'cluster' ] = kmeans_model.fit_predict( X_scaled_domain )

# keep this alias so downstream sections can still use the same naming
domain_input[ 'cluster_domain' ] = domain_input[ 'cluster' ]

domain_silhouette = float( silhouette_score( X_scaled_domain, domain_input[ 'cluster' ] ) )

print( f'\nchosen K: { k_clusters }' )
print( 'domain-feature silhouette:', round( domain_silhouette, 4 ) )

print( '\ncluster sizes:' )
print( domain_input[ 'cluster' ].value_counts( ).sort_index( ) )

cluster_profile_raw = domain_input.groupby( 'cluster' )[ domain_feature_cols ].mean( )
cluster_profile = cluster_profile_raw.round( 3 )

cluster_profile_mean = cluster_profile_raw.mean( )
cluster_profile_std = cluster_profile_raw.std( ddof = 0 ).replace( 0, np.nan )
cluster_profile_z = ( cluster_profile_raw - cluster_profile_mean ) / cluster_profile_std


def bucket_tag( value, low_label, mid_label, high_label, threshold = 0.35 ):
    if pd.isna( value ):
        return mid_label

    if value >= threshold:
        return high_label

    if value <= -threshold:
        return low_label

    return mid_label


def build_cluster_name( z_row ):
    temp_tag = bucket_tag( z_row[ 'mean_annual_water_temp' ], 'Cool', 'Temp', 'Warm' )
    sal_tag = bucket_tag( z_row[ 'mean_annual_salinity' ], 'Fresh', 'Brackish', 'Saline' )
    oxy_tag = bucket_tag( z_row[ 'mean_annual_oxygen' ], 'Low O2', 'Mid O2', 'High O2' )

    amp_tag = bucket_tag( z_row[ 'seasonal_amp_water_temp' ], 'Stable', 'Mixed', 'Seasonal' )
    depth_tag = bucket_tag( z_row[ 'mean_annual_depth' ], 'Shallow', 'Mid', 'Deep' )

    name = f'{ temp_tag } / { sal_tag } / { oxy_tag }'
    short_name = f'{ temp_tag }-{ sal_tag }'
    note = f'{ amp_tag }, { depth_tag }'

    return pd.Series( { 'cluster_name': name, 'cluster_label': short_name, 'cluster_note': note } )


cluster_name_map = cluster_profile_z.apply( build_cluster_name, axis = 1 ).reset_index( )

print( '\ncluster names:' )
print( cluster_name_map )



# keep ordered cluster labels around because phase 9 will export them
cluster_order = sorted( cluster_name_map[ "cluster" ].astype( int ).tolist( ) )
cluster_name_order = list( dict.fromkeys( cluster_name_map.sort_values( "cluster" )[ "cluster_name" ].tolist( ) ) )
cluster_label_order = list( dict.fromkeys( cluster_name_map.sort_values( "cluster" )[ "cluster_label" ].tolist( ) ) )
cluster_note_order = list( dict.fromkeys( cluster_name_map.sort_values( "cluster" )[ "cluster_note" ].tolist( ) ) )

domain_input = domain_input.merge( cluster_name_map, on = "cluster", how = "left" )
cluster_cols = [ "cluster", "cluster_domain", "cluster_name", "cluster_label", "cluster_note" ]
station_cluster_labels = domain_input[ [ "region", "station" ] + cluster_cols ]

station_baseline = ( 
    station_baseline
    .drop( columns = cluster_cols, errors = "ignore" )
    .merge( station_cluster_labels, on = [ "region", "station" ], how = "left" )
)
station_baseline_display = ( 
    station_baseline_display
    .drop( columns = cluster_cols, errors = "ignore" )
    .merge( station_cluster_labels, on = [ "region", "station" ], how = "left" )
)

station_baseline[ "cluster" ] = pd.Categorical( 
    station_baseline[ "cluster" ],
    categories = cluster_order,
    ordered = True,
)
station_baseline[ "cluster_name" ] = pd.Categorical( 
    station_baseline[ "cluster_name" ],
    categories = cluster_name_order,
    ordered = True,
)
station_baseline[ "cluster_label" ] = pd.Categorical( 
    station_baseline[ "cluster_label" ],
    categories = cluster_label_order,
    ordered = True,
)

cluster_station = station_baseline_display

print( f"chosen regime count: { len( cluster_order ) }" )
display( cluster_name_map.sort_values( "cluster" ) )


## Daily Features

Intent:
- roll the atmospheric drivers down to daily features
- compute station-relative water deltas that the later models actually predict

Result to expect:
- `daily_air`
- `daily_water`
- `properties_baseline`


In [ ]:
# and now, let's roll the main driver table down to daily rows
water[ "datetime" ] = pd.to_datetime( water[ "datetime" ], errors = "coerce" )
phase3 = water.dropna( subset = [ "datetime" ] )
phase3[ "date" ] = phase3[ "datetime" ].dt.floor( "D" )

daily_air = ( 
    phase3
    .groupby( [ "region", "station", "date" ], as_index = False )
    .agg( 
        air_temp = ( "air_temp", "mean" ),
        wind_speed = ( "wind_speed", "mean" ),
        solar = ( "solar_radiation", "mean" ),
        precip = ( "precipitation", "sum" ),
    )
    .sort_values( [ "region", "station", "date" ] )
)

# and now, let's loop through the drivers and make short, medium, and longer rolls
for source_col in [ "air_temp", "wind_speed", "precip", "solar" ]:
    for window in [ 1, 7, 28 ]:
        daily_air[ f"{ source_col }_r{ window }d" ] = ( 
            daily_air
            .groupby( [ "region", "station" ] )[ source_col ]
            .transform( lambda values: values.shift( 1 ).rolling( window = window, min_periods = 1 ).mean( ) )
        )

# 0.25 includes leap days
daily_air[ "doy_sin" ] = np.sin( 2 * np.pi * daily_air[ "date" ].dt.dayofyear / 365.25 )
daily_air[ "doy_cos" ] = np.cos( 2 * np.pi * daily_air[ "date" ].dt.dayofyear / 365.25 )

# and now, let's build the daily water table that phase 5, 6, 7, and 9 all share
baseline_start = "1995-01-01"
baseline_end = "2000-12-31"

daily_water = ( 
    phase3
    .groupby( [ "region", "station", "date" ], as_index = False )
    .agg( 
        water_temp = ( "water_temp", "mean" ),
        salinity = ( "salinity", "mean" ),
        oxygen = ( "oxygen", "mean" ),
        ph = ( "ph", "mean" ),
        depth = ( "depth", "mean" ),
    )
    .sort_values( [ "region", "station", "date" ] )
)

properties_baseline = ( 
    daily_water
    .loc[ daily_water[ "date" ].between( baseline_start, baseline_end ) ]
    .groupby( [ "region", "station" ], as_index = False )
    .agg( 
        water_temp_baseline = ( "water_temp", "mean" ),
        salinity_baseline = ( "salinity", "mean" ),
        oxygen_baseline = ( "oxygen", "mean" ),
        ph_baseline = ( "ph", "mean" ),
        depth_baseline = ( "depth", "mean" ),
    )
)

daily_water = daily_water.merge( properties_baseline, on = [ "region", "station" ], how = "left" )
daily_water[ "delta_water_temp" ] = daily_water[ "water_temp" ] - daily_water[ "water_temp_baseline" ]
daily_water[ "delta_salinity" ] = daily_water[ "salinity" ] - daily_water[ "salinity_baseline" ]
daily_water[ "delta_oxygen" ] = daily_water[ "oxygen" ] - daily_water[ "oxygen_baseline" ]
daily_water[ "delta_ph" ] = daily_water[ "ph" ] - daily_water[ "ph_baseline" ]
daily_water[ "delta_depth" ] = daily_water[ "depth" ] - daily_water[ "depth_baseline" ]

print( f"daily_air rows: { len( daily_air ) }" )
print( f"daily_water rows: { len( daily_water ) }" )
print( f"baseline property rows: { len( properties_baseline ) }" )


## Quality Screens And Cohorts

Intent:
- keep only stations and years that can actually support modeling
- detect stations that appear to transition away from their baseline regime
- use those transition stations as the holdout group

Result to expect:
- `seasonal_break_station_keys`
- `pointless_station_keys`
- `first_event`
- final `*_train_final`, `*_holdout_final`, and `*_final` tables


In [ ]:
# and now, let's find recurring same-time seasonal gaps because they can fake transitions
phase2_out_dir = ref_dir / "phase2_lite"
phase2_out_dir.mkdir( parents = True, exist_ok = True )

recurring_break_min_full_years = 3
recurring_break_min_missing_run_months = 2
recurring_break_year_share_threshold = 0.50
recurring_break_same_time_share_threshold = 0.60

daily_presence = daily_air[ [ "region", "station", "date" ] ].drop_duplicates( )
daily_presence[ "year" ] = daily_presence[ "date" ].dt.year
daily_presence[ "month" ] = daily_presence[ "date" ].dt.month


def first_full_calendar_year( ts ):
    if pd.isna( ts ):
        return np.nan

    if ts.month == 1 and ts.day == 1:
        return int( ts.year )

    return int( ts.year ) + 1


def last_full_calendar_year( ts ):
    if pd.isna( ts ):
        return np.nan

    if ts.month == 12 and ts.day == 31:
        return int( ts.year )

    return int( ts.year ) - 1


def summarize_missing_month_run( month_obs_values ):
    max_run = 0
    best_start_month = np.nan
    current_run = 0
    current_start = None

    for month_idx, obs_days in enumerate( month_obs_values, start = 1 ):
        if obs_days == 0:
            if current_run == 0:
                current_start = month_idx
            current_run += 1
            if current_run > max_run:
                max_run = current_run
                best_start_month = current_start

        else:
            current_run = 0
            current_start = None

    if max_run < recurring_break_min_missing_run_months:
        best_start_month = np.nan

    return pd.Series( 
        { 
            "max_consecutive_missing_months": int( max_run ),
            "long_gap_start_month": best_start_month,
            "has_long_gap": bool( max_run >= recurring_break_min_missing_run_months ),
        }
    )


def summarize_station_year_gap( frame ):
    frame = frame.sort_values( "month" )
    run_summary = summarize_missing_month_run( frame[ "n_obs_days" ].tolist( ) )
    return pd.Series( 
        { 
            "n_missing_whole_months": int( frame[ "missing_whole_month" ].sum( ) ),
            "n_obs_months": int( ( ~frame[ "missing_whole_month" ] ).sum( ) ),
            "max_consecutive_missing_months": int( run_summary[ "max_consecutive_missing_months" ] ),
            "long_gap_start_month": run_summary[ "long_gap_start_month" ],
            "has_long_gap": bool( run_summary[ "has_long_gap" ] ),
        }
    )


station_span_p2 = ( 
    daily_presence
    .groupby( [ "region", "station" ], as_index = False )
    .agg( 
        first_obs_date = ( "date", "min" ),
        last_obs_date = ( "date", "max" ),
        n_years_obs = ( "year", "nunique" ),
    )
)
station_span_p2[ "first_full_year" ] = station_span_p2[ "first_obs_date" ].apply( first_full_calendar_year )
station_span_p2[ "last_full_year" ] = station_span_p2[ "last_obs_date" ].apply( last_full_calendar_year )
station_span_p2[ "n_full_years" ] = np.maximum( 
    0,
    station_span_p2[ "last_full_year" ] - station_span_p2[ "first_full_year" ] + 1,
).astype( int )

monthly_obs_counts_p2 = ( 
    daily_presence
    .groupby( [ "region", "station", "year", "month" ], as_index = False )
    .agg( n_obs_days = ( "date", "nunique" ) )
)

full_year_rows = [ ]
for row in station_span_p2.itertuples( index = False ):
    if int( row.n_full_years ) <= 0:
        continue

    for year in range( int( row.first_full_year ), int( row.last_full_year ) + 1 ):
        full_year_rows.append( 
            { 
                "region": row.region,
                "station": row.station,
                "year": int( year ),
            }
        )

if full_year_rows:
    station_full_years_p2 = pd.DataFrame( full_year_rows )
    month_grid = pd.DataFrame( { "month": np.arange( 1, 13, dtype = int ) } )
    station_full_years_p2[ "_tmp" ] = 1
    month_grid[ "_tmp" ] = 1
    station_year_month_grid_p2 = station_full_years_p2.merge( month_grid, on = "_tmp", how = "inner" ).drop( columns = [ "_tmp" ] )

else:
    station_year_month_grid_p2 = pd.DataFrame( columns = [ "region", "station", "year", "month" ] )

station_year_month_p2 = station_year_month_grid_p2.merge( 
    monthly_obs_counts_p2,
    on = [ "region", "station", "year", "month" ],
    how = "left",
)
station_year_month_p2[ "n_obs_days" ] = station_year_month_p2[ "n_obs_days" ].fillna( 0 ).astype( int )
station_year_month_p2[ "missing_whole_month" ] = station_year_month_p2[ "n_obs_days" ] == 0

station_year_gap_summary_p2 = ( 
    station_year_month_p2
    .groupby( [ "region", "station", "year" ] )
    .apply( summarize_station_year_gap )
    .reset_index( )
    if len( station_year_month_p2 ) > 0
    else pd.DataFrame( 
        columns = [ 
            "region",
            "station",
            "year",
            "n_missing_whole_months",
            "n_obs_months",
            "max_consecutive_missing_months",
            "long_gap_start_month",
            "has_long_gap",
        ]
    )
)

long_gap_years_p2 = station_year_gap_summary_p2.loc[ station_year_gap_summary_p2[ "has_long_gap" ] ]
long_gap_mode_p2 = ( 
    long_gap_years_p2
    .groupby( [ "region", "station", "long_gap_start_month" ], as_index = False )
    .agg( n_years_same_start = ( "year", "nunique" ) )
    .sort_values( [ "region", "station", "n_years_same_start", "long_gap_start_month" ], ascending = [ True, True, False, True ] )
    .drop_duplicates( subset = [ "region", "station" ] )
    .rename( columns = { "long_gap_start_month": "modal_long_gap_start_month" } )
    if len( long_gap_years_p2 ) > 0
    else pd.DataFrame( columns = [ "region", "station", "modal_long_gap_start_month", "n_years_same_start" ] )
)

seasonal_break_summary = station_span_p2.merge( 
    ( 
        station_year_gap_summary_p2
        .groupby( [ "region", "station" ], as_index = False )
        .agg( 
            median_missing_whole_months = ( "n_missing_whole_months", "median" ),
            median_max_consecutive_missing_months = ( "max_consecutive_missing_months", "median" ),
            n_full_years_long_gap = ( "has_long_gap", "sum" ),
            pct_full_years_with_long_gap = ( "has_long_gap", "mean" ),
        )
    ),
    on = [ "region", "station" ],
    how = "left",
)
seasonal_break_summary = seasonal_break_summary.merge( long_gap_mode_p2, on = [ "region", "station" ], how = "left" )

for col in [ 
    "median_missing_whole_months",
    "median_max_consecutive_missing_months",
    "n_full_years_long_gap",
    "pct_full_years_with_long_gap",
    "n_years_same_start",
]:
    seasonal_break_summary[ col ] = seasonal_break_summary[ col ].fillna( 0 )

seasonal_break_summary[ "modal_long_gap_start_month" ] = seasonal_break_summary[ "modal_long_gap_start_month" ].fillna( 0 ).astype( int )
seasonal_break_summary[ "modal_long_gap_start_share" ] = np.where( 
    seasonal_break_summary[ "n_full_years_long_gap" ] > 0,
    seasonal_break_summary[ "n_years_same_start" ] / seasonal_break_summary[ "n_full_years_long_gap" ],
    0.0,
)
seasonal_break_summary[ "seasonal_break_station" ] = ( 
    ( seasonal_break_summary[ "n_full_years" ] >= recurring_break_min_full_years )
    & ( seasonal_break_summary[ "pct_full_years_with_long_gap" ] >= recurring_break_year_share_threshold )
    & ( seasonal_break_summary[ "modal_long_gap_start_share" ] >= recurring_break_same_time_share_threshold )
)
seasonal_break_station_keys = ( 
    seasonal_break_summary
    .loc[ seasonal_break_summary[ "seasonal_break_station" ], [ "region", "station" ] ]
    .drop_duplicates( )
    .sort_values( [ "region", "station" ] )
    .reset_index( drop = True )
)

# and now, let's mark stations that were lookup-preexcluded or are just too short-lived to matter here
pointless_station_min_record_span_years = 15.0
pointless_core_cols = [ col for col in [ "water_temp", "salinity", "oxygen", "ph", "depth" ] if col in water.columns ]
pointless_station_source = water[ [ "region", "station", "datetime" ] + pointless_core_cols ]
pointless_station_source[ "timestamp" ] = pd.to_datetime( pointless_station_source[ "datetime" ], errors = "coerce" )
pointless_station_source = pointless_station_source.dropna( subset = [ "timestamp" ] )
pointless_station_source[ "date" ] = pointless_station_source[ "timestamp" ].dt.floor( "D" )
pointless_station_source[ "year" ] = pointless_station_source[ "timestamp" ].dt.year
pointless_station_source[ "has_core_signal" ] = pointless_station_source[ pointless_core_cols ].notna( ).any( axis = 1 )
pointless_station_source[ "core_date" ] = pointless_station_source[ "date" ].where( pointless_station_source[ "has_core_signal" ] )
pointless_station_source[ "core_year" ] = pointless_station_source[ "year" ].where( pointless_station_source[ "has_core_signal" ] )

pointless_station_history = ( 
    pointless_station_source
    .groupby( [ "region", "station" ], as_index = False )
    .agg( 
        n_raw_rows = ( "datetime", "size" ),
        n_raw_dates = ( "date", "nunique" ),
        n_raw_years = ( "year", "nunique" ),
        n_core_rows = ( "has_core_signal", "sum" ),
        n_core_dates = ( "core_date", "nunique" ),
        n_core_years = ( "core_year", "nunique" ),
        first_obs_date = ( "date", "min" ),
        last_obs_date = ( "date", "max" ),
    )
)
pointless_station_history[ "record_span_days" ] = ( 
    pointless_station_history[ "last_obs_date" ] - pointless_station_history[ "first_obs_date" ]
).dt.days.fillna( 0 )
pointless_station_history[ "record_span_years" ] = ( pointless_station_history[ "record_span_days" ] / 365.25 ).round( 4 )

pointless_station_summary = station_lookup[ 
    [ "region", "station", "region_name", "station_name", "in_t4d_1hr_water_history" ]
].drop_duplicates( subset = [ "region", "station" ] ).merge( 
    pointless_station_history,
    on = [ "region", "station" ],
    how = "left",
)
for col in [ "n_raw_rows", "n_raw_dates", "n_raw_years", "n_core_rows", "n_core_dates", "n_core_years", "record_span_days" ]:
    pointless_station_summary[ col ] = pointless_station_summary[ col ].fillna( 0 ).astype( int )
pointless_station_summary[ "record_span_years" ] = pointless_station_summary[ "record_span_years" ].fillna( 0.0 )
pointless_station_summary[ "lookup_preexcluded" ] = ~pointless_station_summary[ "in_t4d_1hr_water_history" ].fillna( False ).astype( bool )
pointless_station_summary[ "short_record_span_station" ] = pointless_station_summary[ "record_span_years" ] < pointless_station_min_record_span_years
pointless_station_summary[ "pointless_station" ] = ( 
    pointless_station_summary[ "lookup_preexcluded" ]
    | pointless_station_summary[ "short_record_span_station" ]
)
pointless_station_keys = pointless_station_summary.loc[ 
    pointless_station_summary[ "pointless_station" ],
    [ "region", "station" ],
].drop_duplicates( )

print( f"seasonal break stations: { len( seasonal_break_station_keys ) }" )
print( f"pointless stations: { len( pointless_station_keys ) }" )


In [ ]:
# start from daily_water, then build monthly summaries for more gap-tolerant rolling windows
fiveyear_water = daily_water.copy( )
fiveyear_water[ 'date' ] = pd.to_datetime( fiveyear_water[ 'date' ], errors = 'coerce' )
fiveyear_water = fiveyear_water.dropna( subset = [ 'date' ] )
fiveyear_water = fiveyear_water.sort_values( [ 'region', 'station', 'date' ] )
fiveyear_water[ 'month' ] = fiveyear_water[ 'date' ].dt.to_period( 'M' ).dt.to_timestamp( )

# keep this explicit and readable for students
water_metric_candidates = [ 
    'water_temp',
    'salinity',
    'oxygen',
    'ph',
    'depth',
]

# only use metrics that are actually present in daily_water
fiveyear_water_metrics = [ m for m in water_metric_candidates if m in fiveyear_water.columns ]

monthly_water = ( 
    fiveyear_water
    .groupby( [ 'region', 'station', 'month' ], as_index = False )[ fiveyear_water_metrics ]
    .mean( )
    .sort_values( [ 'region', 'station', 'month' ] )
)

print( 'monthly metrics used for rolling windows:', fiveyear_water_metrics )

# 5-year rolling means on monthly data (60 months)
# this is more tolerant to seasonal gaps than strict daily rolling
for metric in fiveyear_water_metrics:
    monthly_water[ f'{metric}_roll5y' ] = ( 
        monthly_water
        .groupby( [ 'region', 'station' ] )[ metric ]
        .transform( lambda values: values.rolling( window = 60, min_periods = 36 ).mean( ) )
    )

# rolling coverage over the same 5-year monthly window
monthly_water[ 'has_any_data' ] = monthly_water[ fiveyear_water_metrics ].notna( ).any( axis = 1 ).astype( int )
monthly_water[ 'months_with_any_data_5y' ] = ( 
    monthly_water
    .groupby( [ 'region', 'station' ] )[ 'has_any_data' ]
    .transform( lambda values: values.rolling( window = 60, min_periods = 1 ).sum( ) )
)
monthly_water[ 'coverage_ratio_5y' ] = monthly_water[ 'months_with_any_data_5y' ] / 60.0

# attach monthly rolling outputs back onto daily rows for plotting and event timing
roll5y_cols = [ f'{m}_roll5y' for m in fiveyear_water_metrics ]
monthly_roll_cols = [ 'region', 'station', 'month' ] + roll5y_cols + [ 'months_with_any_data_5y', 'coverage_ratio_5y' ]
fiveyear_water = fiveyear_water.merge( monthly_water[ monthly_roll_cols ], on = [ 'region', 'station', 'month' ], how = 'left' )

# keep a backward-compatible alias in case later cells use "fiveyear"
fiveyear = fiveyear_water.copy( )

# build a monthly Phase-5 working table for classification
required_roll_cols = [ 
    'water_temp_roll5y',
    'salinity_roll5y',
    'oxygen_roll5y',
    'depth_roll5y',
]

available_roll_cols = [ col for col in required_roll_cols if col in monthly_water.columns ]

p5_monthly = monthly_water[ [ 'region', 'station', 'month', 'coverage_ratio_5y' ] + available_roll_cols ].copy( )

p5_monthly = p5_monthly.rename( columns = { 
    'month': 'date',
    'water_temp_roll5y': 'mean_annual_water_temp',
    'salinity_roll5y': 'mean_annual_salinity',
    'oxygen_roll5y': 'mean_annual_oxygen',
    'depth_roll5y': 'mean_annual_depth',
} )

# attach baseline cluster identity
baseline_cluster = cluster_station[ [ 'region', 'station', 'cluster' ] ].drop_duplicates( )
p5_monthly = p5_monthly.merge( baseline_cluster, on = [ 'region', 'station' ], how = 'left' )

# compare against baseline centroids in standardized feature space
feature_cols = [ 
    'mean_annual_water_temp',
    'mean_annual_salinity',
    'mean_annual_oxygen',
    'mean_annual_depth',
]

# baseline station profiles from phase 1
ref = station_baseline[ [ 'cluster' ] + feature_cols ].dropna( ).copy( )
scaler_p5 = StandardScaler( ).fit( ref[ feature_cols ] )

ref_z = pd.DataFrame( 
    scaler_p5.transform( ref[ feature_cols ] ),
    columns = feature_cols,
    index = ref.index,
)
ref_z[ 'cluster' ] = ref[ 'cluster' ].values

centroids_z = ( 
    ref_z
    .groupby( 'cluster', as_index = True )[ feature_cols ]
    .mean( )
    .sort_index( )
)

# monthly nearest-centroid assignment with coverage + partial-feature tolerance
coverage_threshold = 0.70
min_features_required = 3

feature_mean = pd.Series( scaler_p5.mean_, index = feature_cols )
feature_scale = pd.Series( scaler_p5.scale_, index = feature_cols )


def classify_month_row( row ):
    coverage = row.get( 'coverage_ratio_5y', np.nan )
    if pd.isna( coverage ) or coverage < coverage_threshold:
        return pd.Series( { 
            'implied_cluster': np.nan,
            'dist_best': np.nan,
            'dist_second': np.nan,
            'margin': np.nan,
            'n_features_used': 0,
            'assignment_state': 'insufficient_coverage',
        } )

    available = [ col for col in feature_cols if pd.notna( row.get( col, np.nan ) ) ]
    if len( available ) < min_features_required:
        return pd.Series( { 
            'implied_cluster': np.nan,
            'dist_best': np.nan,
            'dist_second': np.nan,
            'margin': np.nan,
            'n_features_used': len( available ),
            'assignment_state': 'insufficient_features',
        } )

    values = pd.to_numeric( row[ available ], errors = 'coerce' )
    scales = pd.to_numeric( feature_scale[ available ], errors = 'coerce' ).replace( 0, np.nan )
    means = pd.to_numeric( feature_mean[ available ], errors = 'coerce' )

    z_values = ( values - means ) / scales
    if z_values.isna( ).any( ):
        return pd.Series( { 
            'implied_cluster': np.nan,
            'dist_best': np.nan,
            'dist_second': np.nan,
            'margin': np.nan,
            'n_features_used': len( available ),
            'assignment_state': 'scaling_issue',
        } )

    centroid_slice = centroids_z[ available ]

    try:
        centroid_array = np.asarray( centroid_slice.to_numpy( ), dtype = float )
        z_array = np.asarray( z_values.to_numpy( ), dtype = float )
        distances = np.sqrt( np.sum( ( centroid_array - z_array[ None, : ] ) ** 2, axis = 1 ) )

    except Exception:
        return pd.Series( { 
            'implied_cluster': np.nan,
            'dist_best': np.nan,
            'dist_second': np.nan,
            'margin': np.nan,
            'n_features_used': len( available ),
            'assignment_state': 'distance_error',
        } )

    best_idx = int( np.argmin( distances ) )
    best_cluster = int( centroid_slice.index[ best_idx ] )
    best_dist = float( distances[ best_idx ] )

    if len( distances ) > 1:
        second_dist = float( np.partition( distances, 1 )[ 1 ] )
        margin = second_dist - best_dist

    else:
        second_dist = np.nan
        margin = np.nan

    state = 'classified_full' if len( available ) == len( feature_cols ) else 'classified_partial'

    return pd.Series( { 
        'implied_cluster': best_cluster,
        'dist_best': best_dist,
        'dist_second': second_dist,
        'margin': margin,
        'n_features_used': len( available ),
        'assignment_state': state,
    } )

# apply classification row-by-row after coercing feature columns to numeric
for col in feature_cols:
    if col in p5_monthly.columns:
        p5_monthly[ col ] = pd.to_numeric( p5_monthly[ col ], errors = 'coerce' )

p5_monthly = pd.concat( [ p5_monthly, p5_monthly.apply( classify_month_row, axis = 1 ) ], axis = 1 )

p5_monthly[ 'assignment_state' ].value_counts( dropna = False )

# confirm transitions on monthly assignments, then project onto daily rows
p5_monthly = p5_monthly.sort_values( [ 'region', 'station', 'date' ] ).copy( )

p5_monthly[ 'candidate_flip' ] = ( 
    p5_monthly[ 'implied_cluster' ].notna( )
    & p5_monthly[ 'cluster' ].notna( )
    & ( p5_monthly[ 'implied_cluster' ] != p5_monthly[ 'cluster' ] )
)

monthly_run_id = ( 
    p5_monthly
    .groupby( [ 'region', 'station' ] )[ 'implied_cluster' ]
    .transform( lambda values: values.ne( values.shift( ) ).cumsum( ) )
)

p5_monthly[ 'run_len_months' ] = ( 
    p5_monthly
    .groupby( [ 'region', 'station', monthly_run_id ] )[ 'implied_cluster' ]
    .transform( 'size' )
)

p5_monthly[ 'flip_confirmed_monthly' ] = ( 
    p5_monthly[ 'candidate_flip' ]
    & ( p5_monthly[ 'run_len_months' ] >= 6 )   # ~6 months persistence
    & ( p5_monthly[ 'margin' ] > 0.10 )
)

# bring monthly classification decisions down to each daily observation row
p5 = fiveyear_water[ [ 'region', 'station', 'date', 'month' ] ].copy( )

monthly_decision_cols = [ 
    'region',
    'station',
    'date',
    'cluster',
    'implied_cluster',
    'dist_best',
    'dist_second',
    'margin',
    'n_features_used',
    'assignment_state',
    'coverage_ratio_5y',
    'candidate_flip',
    'run_len_months',
    'flip_confirmed_monthly',
]

p5 = p5.merge( 
    p5_monthly[ monthly_decision_cols ].rename( columns = { 'date': 'month' } ),
    on = [ 'region', 'station', 'month' ],
    how = 'left',
)

p5[ 'flip_confirmed' ] = p5[ 'flip_confirmed_monthly' ].fillna( False ).astype( bool )
p5[ 'run_len_days' ] = p5[ 'run_len_months' ] * 30.4375

p5 = p5.sort_values( [ 'region', 'station', 'date' ] ).copy( )

# anything?
first_event = ( 
    p5.loc[ p5[ 'flip_confirmed' ] ]
    .groupby( [ 'region', 'station' ], as_index = False )[ 'date' ]
    .min( )
    .rename( columns = { 'date': 'event_date' } )
)

first_event


In [ ]:
# and now, let's clean the modeling scope and split train from holdout
core_water_cols = [ col for col in [ "water_temp", "salinity", "oxygen", "ph", "depth" ] if col in daily_water.columns ]

station_raw_quality = ( 
    daily_water
    .groupby( [ "region", "station" ], as_index = False )
    .agg( n_daily_rows = ( "date", "size" ) )
)
raw_non_null_counts = ( 
    daily_water
    .groupby( [ "region", "station" ] )[ core_water_cols ]
    .apply( lambda frame: int( frame.notna( ).sum( ).sum( ) ) )
    .rename( "n_non_null_core_water" )
    .reset_index( )
)
station_raw_quality = station_raw_quality.merge( raw_non_null_counts, on = [ "region", "station" ], how = "left" )
station_raw_quality[ "has_raw_signal" ] = station_raw_quality[ "n_non_null_core_water" ] > 0

valid_station_keys = station_raw_quality.loc[ station_raw_quality[ "has_raw_signal" ], [ "region", "station" ] ].drop_duplicates( )
valid_station_keys = ( 
    valid_station_keys
    .merge( pointless_station_keys.assign( has_pointless_station = True ), on = [ "region", "station" ], how = "left" )
    .merge( seasonal_break_station_keys.assign( has_seasonal_break = True ), on = [ "region", "station" ], how = "left" )
    .loc[ 
        lambda frame: ~frame[ "has_pointless_station" ].fillna( False ).astype( bool )
        & ~frame[ "has_seasonal_break" ].fillna( False ).astype( bool ),
        [ "region", "station" ],
    ]
    .drop_duplicates( )
)


def keep_valid_stations( frame ):
    return frame.merge( valid_station_keys, on = [ "region", "station" ], how = "inner" )


daily_air_model = keep_valid_stations( daily_air )
daily_water_model = keep_valid_stations( daily_water )
fiveyear_water_model = keep_valid_stations( fiveyear_water )
p5_model = keep_valid_stations( p5 )
p5_monthly_model = keep_valid_stations( p5_monthly )

# and now, let's do the station-year screen that phase 6 and phase 7 will both share
station_year_min_any_core_days = 120
station_year_min_temp_days = 120
station_year_min_temp_months = 6
station_year_min_warm_oxygen_days = 45
station_year_min_warm_oxygen_months = 3

station_year_source = daily_water_model
station_year_source[ "date" ] = pd.to_datetime( station_year_source[ "date" ], errors = "coerce" )
station_year_source[ "year" ] = station_year_source[ "date" ].dt.year
station_year_source[ "month_num" ] = station_year_source[ "date" ].dt.month
station_year_source[ "has_any_core_obs" ] = station_year_source[ core_water_cols ].notna( ).any( axis = 1 )
station_year_source[ "has_temp_obs" ] = station_year_source[ "water_temp" ].notna( )
station_year_source[ "has_oxygen_obs" ] = station_year_source[ "oxygen" ].notna( )

station_year_quality = ( 
    station_year_source
    .groupby( [ "region", "station", "year" ], as_index = False )
    .agg( 
        n_days = ( "date", "size" ),
        n_any_core_days = ( "has_any_core_obs", "sum" ),
        n_temp_days = ( "has_temp_obs", "sum" ),
        n_oxygen_days = ( "has_oxygen_obs", "sum" ),
    )
)

month_any = station_year_source.loc[ station_year_source[ "has_any_core_obs" ], [ "region", "station", "year", "month_num" ] ].drop_duplicates( )
month_temp = station_year_source.loc[ station_year_source[ "has_temp_obs" ], [ "region", "station", "year", "month_num" ] ].drop_duplicates( )
month_oxygen = station_year_source.loc[ station_year_source[ "has_oxygen_obs" ], [ "region", "station", "year", "month_num" ] ].drop_duplicates( )

for out_col, source_df in [ 
    ( "n_any_core_months", month_any ),
    ( "n_temp_months", month_temp ),
    ( "n_oxygen_months", month_oxygen ),
]:
    month_counts = source_df.groupby( [ "region", "station", "year" ] ).size( ).rename( out_col ).reset_index( )
    station_year_quality = station_year_quality.merge( month_counts, on = [ "region", "station", "year" ], how = "left" )

warm_source = station_year_source.loc[ station_year_source[ "month_num" ].between( 6, 9 ) ]
warm_quality = ( 
    warm_source
    .groupby( [ "region", "station", "year" ], as_index = False )
    .agg( 
        n_warm_days = ( "date", "size" ),
        n_warm_temp_days = ( "has_temp_obs", "sum" ),
        n_warm_oxygen_days = ( "has_oxygen_obs", "sum" ),
    )
)

warm_month_temp = warm_source.loc[ warm_source[ "has_temp_obs" ], [ "region", "station", "year", "month_num" ] ].drop_duplicates( )
warm_month_oxygen = warm_source.loc[ warm_source[ "has_oxygen_obs" ], [ "region", "station", "year", "month_num" ] ].drop_duplicates( )

for out_col, source_df in [ 
    ( "n_warm_temp_months", warm_month_temp ),
    ( "n_warm_oxygen_months", warm_month_oxygen ),
]:
    month_counts = source_df.groupby( [ "region", "station", "year" ] ).size( ).rename( out_col ).reset_index( )
    warm_quality = warm_quality.merge( month_counts, on = [ "region", "station", "year" ], how = "left" )

station_year_quality = station_year_quality.merge( warm_quality, on = [ "region", "station", "year" ], how = "left" ).fillna( 0 )
station_year_quality[ "has_model_year" ] = ( 
    ( station_year_quality[ "n_any_core_days" ] >= station_year_min_any_core_days )
    & ( station_year_quality[ "n_temp_days" ] >= station_year_min_temp_days )
    & ( station_year_quality[ "n_temp_months" ] >= station_year_min_temp_months )
)
station_year_quality[ "has_warm_oxygen_year" ] = ( 
    station_year_quality[ "has_model_year" ]
    & ( station_year_quality[ "n_warm_oxygen_days" ] >= station_year_min_warm_oxygen_days )
    & ( station_year_quality[ "n_warm_oxygen_months" ] >= station_year_min_warm_oxygen_months )
)


def keep_valid_station_years( frame, date_col = "date", flag_col = "has_model_year" ):
    actual_date_col = date_col if date_col in frame.columns else "month"
    dated = frame.dropna( subset = [ actual_date_col ] )
    dated[ actual_date_col ] = pd.to_datetime( dated[ actual_date_col ], errors = "coerce" )
    dated[ "year" ] = dated[ actual_date_col ].dt.year
    keep_keys = station_year_quality.loc[ 
        station_year_quality[ flag_col ].fillna( False ).astype( bool ),
        [ "region", "station", "year" ],
    ].drop_duplicates( )
    return dated.merge( keep_keys, on = [ "region", "station", "year" ], how = "inner" ).drop( columns = [ "year" ] )


daily_air_model = keep_valid_station_years( daily_air_model )
daily_water_model = keep_valid_station_years( daily_water_model )
fiveyear_water_model = keep_valid_station_years( fiveyear_water_model )
p5_model = keep_valid_station_years( p5_model )
p5_monthly_model = keep_valid_station_years( p5_monthly_model )

transition_station_keys = ( 
    p5_model
    .loc[ p5_model[ "flip_confirmed" ].fillna( False ), [ "region", "station" ] ]
    .drop_duplicates( )
)
holdout_station_keys = transition_station_keys
train_station_keys = ( 
    valid_station_keys
    .merge( holdout_station_keys.assign( is_transition = True ), on = [ "region", "station" ], how = "left" )
    .loc[ lambda frame: frame[ "is_transition" ].isna( ), [ "region", "station" ] ]
)

station_classification_quality = ( 
    p5_model
    .groupby( [ "region", "station" ], as_index = False )
    .agg( 
        n_rows = ( "date", "size" ),
        n_classified_days = ( "implied_cluster", lambda values: int( values.notna( ).sum( ) ) ),
        n_flip_days = ( "flip_confirmed", lambda values: int( values.fillna( False ).sum( ) ) ),
    )
)
classifiable_station_keys = station_classification_quality.loc[ 
    station_classification_quality[ "n_classified_days" ] > 0,
    [ "region", "station" ],
].drop_duplicates( )

flagged_station_keys_final = holdout_station_keys.merge( classifiable_station_keys, on = [ "region", "station" ], how = "inner" )
unflagged_station_keys_final = ( 
    classifiable_station_keys
    .merge( flagged_station_keys_final.assign( is_flagged = True ), on = [ "region", "station" ], how = "left" )
    .loc[ lambda frame: frame[ "is_flagged" ].isna( ), [ "region", "station" ] ]
)


def subset_by_station_keys( frame, station_keys ):
    return frame.merge( station_keys, on = [ "region", "station" ], how = "inner" )


daily_air_final = subset_by_station_keys( daily_air_model, classifiable_station_keys )
daily_water_final = subset_by_station_keys( daily_water_model, classifiable_station_keys )
fiveyear_water_final = subset_by_station_keys( fiveyear_water_model, classifiable_station_keys )
p5_final = subset_by_station_keys( p5_model, classifiable_station_keys )
p5_monthly_final = subset_by_station_keys( p5_monthly_model, classifiable_station_keys )

daily_air_train_final = subset_by_station_keys( daily_air_model, unflagged_station_keys_final )
daily_water_train_final = subset_by_station_keys( daily_water_model, unflagged_station_keys_final )
fiveyear_water_train_final = subset_by_station_keys( fiveyear_water_model, unflagged_station_keys_final )
p5_train_final = subset_by_station_keys( p5_model, unflagged_station_keys_final )
p5_monthly_train_final = subset_by_station_keys( p5_monthly_model, unflagged_station_keys_final )

daily_air_holdout_final = subset_by_station_keys( daily_air_model, flagged_station_keys_final )
daily_water_holdout_final = subset_by_station_keys( daily_water_model, flagged_station_keys_final )
fiveyear_water_holdout_final = subset_by_station_keys( fiveyear_water_model, flagged_station_keys_final )
p5_holdout_final = subset_by_station_keys( p5_model, flagged_station_keys_final )
p5_monthly_holdout_final = subset_by_station_keys( p5_monthly_model, flagged_station_keys_final )

print( f"final train stations: { len( unflagged_station_keys_final ) }" )
print( f"final holdout stations: { len( flagged_station_keys_final ) }" )
print( f"transition events found: { len( first_event ) }" )


## Phase 6: Air To Water Temperature

Intent:
- learn the station-relative `delta_water_temp`
- compare a couple of learned models against simple baselines
- keep the best learned model for later scenario runs

Result to expect:
- `phase6_selected_model`
- `phase6_selected_features`
- `phase6_selected_fill_values`
- `phase6_context_table`


### 6.0 Imports And Contract

This cell sets up the phase-6 imports, the static station context table, and the feature lists that the air-to-water model will use.


In [ ]:
# 6.0 - define the context table and the shared feature contract
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold

context_source_water = daily_water_train_final.dropna( subset = [ "date" ] )
context_metric_candidates = [ "water_temp", "salinity", "oxygen", "ph", "depth" ]
context_metrics = [ col for col in context_metric_candidates if col in context_source_water.columns ]

context_agg = { }
for col in context_metrics:
    context_agg[ f"ctx_{ col }_mean" ] = ( col, "mean" )
    context_agg[ f"ctx_{ col }_std" ] = ( col, "std" )

phase6_context_table = ( 
    context_source_water
    .groupby( [ "region", "station" ], as_index = False )
    .agg( **context_agg )
)

phase6_air_feature_candidates = [ 
    "air_temp_r1d",
    "air_temp_r7d",
    "air_temp_r28d",
    "wind_speed_r1d",
    "wind_speed_r7d",
    "wind_speed_r28d",
    "precip_r1d",
    "precip_r7d",
    "precip_r28d",
    "solar_r1d",
    "solar_r7d",
    "solar_r28d",
    "air_temp",
    "wind_speed",
    "precip",
    "solar",
    "doy_sin",
    "doy_cos",
    "air_temp_minus_water_temp_baseline",
    "air_temp_r7d_minus_water_temp_baseline",
    "air_temp_r28d_minus_water_temp_baseline",
    "air_temp_r1d_minus_air_temp_r28d",
    "air_temp_r7d_minus_air_temp_r28d",
    "wind_speed_x_air_temp",
    "solar_x_air_temp",
]

phase6_context_feature_candidates = [ 
    "cluster_code",
    "water_temp_baseline",
    "salinity_baseline",
    "oxygen_baseline",
    "ph_baseline",
    "depth_baseline",
] + [ col for col in phase6_context_table.columns if col.startswith( "ctx_" ) ]


### 6.0 Feature Engineering Helper

This helper adds the interaction and delta features that phase 6 and phase 9 both reuse.


In [ ]:
def add_phase6_engineered_features( frame ):
    frame[ "date" ] = pd.to_datetime( frame[ "date" ], errors = "coerce" )
    frame[ "doy" ] = frame[ "date" ].dt.dayofyear
    frame[ "month" ] = frame[ "date" ].dt.month

    if { "air_temp", "water_temp_baseline" }.issubset( frame.columns ):
        frame[ "air_temp_minus_water_temp_baseline" ] = frame[ "air_temp" ] - frame[ "water_temp_baseline" ]

    if { "air_temp_r7d", "water_temp_baseline" }.issubset( frame.columns ):
        frame[ "air_temp_r7d_minus_water_temp_baseline" ] = frame[ "air_temp_r7d" ] - frame[ "water_temp_baseline" ]

    if { "air_temp_r28d", "water_temp_baseline" }.issubset( frame.columns ):
        frame[ "air_temp_r28d_minus_water_temp_baseline" ] = frame[ "air_temp_r28d" ] - frame[ "water_temp_baseline" ]

    if { "air_temp_r1d", "air_temp_r28d" }.issubset( frame.columns ):
        frame[ "air_temp_r1d_minus_air_temp_r28d" ] = frame[ "air_temp_r1d" ] - frame[ "air_temp_r28d" ]

    if { "air_temp_r7d", "air_temp_r28d" }.issubset( frame.columns ):
        frame[ "air_temp_r7d_minus_air_temp_r28d" ] = frame[ "air_temp_r7d" ] - frame[ "air_temp_r28d" ]

    if { "wind_speed", "air_temp" }.issubset( frame.columns ):
        frame[ "wind_speed_x_air_temp" ] = frame[ "wind_speed" ] * frame[ "air_temp" ]

    if { "solar", "air_temp" }.issubset( frame.columns ):
        frame[ "solar_x_air_temp" ] = frame[ "solar" ] * frame[ "air_temp" ]

    return frame


### 6.0 Calibration Helpers

These helpers keep the model-fitting cell shorter by separating prediction calibration and shared scoring logic.


In [ ]:
class Phase6CalibratedPredictor:


    def __init__( self, base_model, calibrator ):
        self.base_model = base_model
        self.calibrator = calibrator


    def predict( self, X ):
        raw_pred = np.asarray( self.base_model.predict( X ), dtype = "float64" )

        if isinstance( self.calibrator, IsotonicRegression ):
            return self.calibrator.predict( raw_pred )

        return self.calibrator.predict( raw_pred.reshape( -1, 1 ) )


def append_phase6_score( score_rows, model_name, y_true, y_pred ):
    score_rows.append( 
        { 
            "model": model_name,
            "mae": float( mean_absolute_error( y_true, y_pred ) ),
            "rmse": float( mean_squared_error( y_true, y_pred ) ** 0.5 ),
            "r2": float( r2_score( y_true, y_pred ) ),
            "bias": float( np.mean( np.asarray( y_pred ) - np.asarray( y_true ) ) ),
        }
    )


def build_phase6_group_oof_predictions( estimator, X, y, groups ):
    groups = pd.Series( groups, index = X.index ).astype( str )
    unique_groups = groups.dropna( ).unique( )

    if len( unique_groups ) < 2:
        fitted_estimator = clone( estimator )
        fitted_estimator.fit( X, y )
        return pd.Series( fitted_estimator.predict( X ), index = X.index, dtype = "float64" )

    splitter = GroupKFold( n_splits = min( 5, len( unique_groups ) ) )
    oof_pred = pd.Series( index = X.index, dtype = "float64" )

    for fit_idx, valid_idx in splitter.split( X, y, groups ):
        fold_model = clone( estimator )
        fold_model.fit( X.iloc[ fit_idx ], y.iloc[ fit_idx ] )
        oof_pred.iloc[ valid_idx ] = fold_model.predict( X.iloc[ valid_idx ] )

    if oof_pred.isna( ).any( ):
        fallback_model = clone( estimator )
        fallback_model.fit( X, y )
        missing_mask = oof_pred.isna( )
        oof_pred.loc[ missing_mask ] = fallback_model.predict( X.loc[ missing_mask ] )

    return oof_pred


### 6.1 Build Modeling Tables

This cell joins the train and holdout tables, adds station context, and creates the naive baselines.


In [ ]:
# and now, let's build the train and holdout tables
base_join_cols = [ "region", "station", "date" ]
water_keep_cols = [ 
    "delta_water_temp",
    "water_temp_baseline",
    "salinity_baseline",
    "oxygen_baseline",
    "ph_baseline",
    "depth_baseline",
]

phase6_cluster_lookup = station_baseline[ [ "region", "station", "cluster" ] ]
phase6_cluster_lookup[ "cluster_code" ] = pd.to_numeric( phase6_cluster_lookup[ "cluster" ], errors = "coerce" )
phase6_cluster_lookup = phase6_cluster_lookup.drop( columns = [ "cluster" ] )

phase6_train = daily_air_train_final.merge( 
    daily_water_train_final[ base_join_cols + water_keep_cols ],
    on = base_join_cols,
    how = "inner",
)
phase6_holdout = daily_air_holdout_final.merge( 
    daily_water_holdout_final[ base_join_cols + water_keep_cols ],
    on = base_join_cols,
    how = "inner",
)

phase6_train = phase6_train.merge( phase6_context_table, on = [ "region", "station" ], how = "left" )
phase6_holdout = phase6_holdout.merge( phase6_context_table, on = [ "region", "station" ], how = "left" )
phase6_train = phase6_train.merge( phase6_cluster_lookup, on = [ "region", "station" ], how = "left" )
phase6_holdout = phase6_holdout.merge( phase6_cluster_lookup, on = [ "region", "station" ], how = "left" )

phase6_train = add_phase6_engineered_features( phase6_train )
phase6_holdout = add_phase6_engineered_features( phase6_holdout )

target_col = "delta_water_temp"
phase6_train_model = phase6_train.dropna( subset = [ target_col ] )
phase6_holdout_model = phase6_holdout.dropna( subset = [ target_col ] )
phase6_train_model[ "station_group" ] = phase6_train_model[ "region" ].astype( str ) + " | " + phase6_train_model[ "station" ].astype( str )
phase6_holdout_model[ "station_group" ] = phase6_holdout_model[ "region" ].astype( str ) + " | " + phase6_holdout_model[ "station" ].astype( str )

air_features = [ col for col in phase6_air_feature_candidates if col in phase6_train_model.columns ]
context_features = [ col for col in phase6_context_feature_candidates if col in phase6_train_model.columns ]
joint_features = list( dict.fromkeys( air_features + context_features ) )

if len( phase6_train_model ) == 0:
    raise ValueError( "phase 6 has no training rows after cleanup" )

if len( air_features ) == 0:
    raise ValueError( "phase 6 has no air features after cleanup" )

phase6_holdout_model = phase6_holdout_model.sort_values( [ "region", "station", "date" ] )
phase6_holdout_model[ "pred_persistence" ] = phase6_holdout_model.groupby( [ "region", "station" ] )[ target_col ].shift( 1 )

clim_by_doy = ( 
    phase6_train_model
    .groupby( "doy", as_index = False )[ target_col ]
    .median( )
    .rename( columns = { target_col: "pred_climatology" } )
)
phase6_holdout_model[ "pred_climatology" ] = phase6_holdout_model[ "doy" ].map( clim_by_doy.set_index( "doy" )[ "pred_climatology" ] )
global_target_median = float( phase6_train_model[ target_col ].median( ) )
phase6_holdout_model[ "pred_persistence" ] = phase6_holdout_model[ "pred_persistence" ].fillna( phase6_holdout_model[ "pred_climatology" ] ).fillna( global_target_median )
phase6_holdout_model[ "pred_climatology" ] = phase6_holdout_model[ "pred_climatology" ].fillna( global_target_median )

y_train = phase6_train_model[ target_col ]
phase6_scores_rows = [ ]
phase6_calibration_rows = [ ]
phase6_model_store = { }
phase6_feature_store = { }
phase6_fill_store = { }
phase6_pred_col_store = { 
    "naive_persistence": "pred_persistence",
    "naive_climatology_doy": "pred_climatology",
}


### 6.2 Fit Candidate Models

This cell fits the ridge and gradient-boosting candidates, then adds linear and isotonic calibration layers.

**note:** prediction error + alpha * sum( coefficient^2 ) ... akin to linear, but with guardrails

**from reading:**
"*ridge is a sensible model when you have climate and station features that overlap in information*", like:

- air temperature means and standard deviations
- wind, precip, solar summaries
- baseline station traits
- clustered or seasonal descriptors

e.g., "*find the best weights, but don’t trust huge weights unless the data really forces them*"

Not as good at nonlinear behavior and keeps features in the model that might have been selectably out?


In [ ]:
X_train_air = phase6_train_model[ air_features ]
air_fill_values = X_train_air.median( numeric_only = True )
X_train_air = X_train_air.fillna( air_fill_values )

# air-only model as a baseline, since air features are the most common and have the most coverage 
# - let's see how far we can get with just those before even adding context features
ridge_air_model = Ridge( alpha = 1.0 )
ridge_air_model.fit( X_train_air, y_train )
phase6_model_store[ "ridge_air_only" ] = ridge_air_model
phase6_feature_store[ "ridge_air_only" ] = air_features
phase6_fill_store[ "ridge_air_only" ] = air_fill_values
phase6_pred_col_store[ "ridge_air_only" ] = "pred_ridge_air_only"

# we can only build joint models on some rows that have at least some coverage, but let's see if that buys us anything
X_train_joint = phase6_train_model[ joint_features ] if joint_features else pd.DataFrame( index = phase6_train_model.index )
joint_fill_values = X_train_joint.median( numeric_only = True ) if joint_features else pd.Series( dtype = "float64" )
X_train_joint = X_train_joint.fillna( joint_fill_values ) if joint_features else X_train_joint

# note that we don't have enough data to really tune these models, 
# so we'll just pick some reasonable defaults and see how they do 
# - no hyperparameter tuning at this stage
if joint_features:
    ridge_air_context_model = Ridge( alpha = 1.0 )
    ridge_air_context_model.fit( X_train_joint, y_train )
    phase6_model_store[ "ridge_air_context" ] = ridge_air_context_model
    phase6_feature_store[ "ridge_air_context" ] = joint_features
    phase6_fill_store[ "ridge_air_context" ] = joint_fill_values
    phase6_pred_col_store[ "ridge_air_context" ] = "pred_ridge_air_context"

    hgb_air_context_model = HistGradientBoostingRegressor( 
        learning_rate = 0.05,
        max_depth = 6,
        max_iter = 250,
        random_state = 42,
    )
    hgb_air_context_model.fit( X_train_joint, y_train )
    phase6_model_store[ "hgb_air_context" ] = hgb_air_context_model
    phase6_feature_store[ "hgb_air_context" ] = joint_features
    phase6_fill_store[ "hgb_air_context" ] = joint_fill_values
    phase6_pred_col_store[ "hgb_air_context" ] = "pred_hgb_air_context"

if len( phase6_holdout_model ) > 0:
    y_holdout = phase6_holdout_model[ target_col ]
    append_phase6_score( phase6_scores_rows, "naive_persistence", y_holdout, phase6_holdout_model[ "pred_persistence" ] )
    append_phase6_score( phase6_scores_rows, "naive_climatology_doy", y_holdout, phase6_holdout_model[ "pred_climatology" ] )

for model_name in [ "ridge_air_only", "ridge_air_context", "hgb_air_context" ]:
    estimator = phase6_model_store.get( model_name )
    feature_cols = phase6_feature_store.get( model_name, [ ] )
    fill_values = phase6_fill_store.get( model_name, pd.Series( dtype = "float64" ) )
    pred_col = phase6_pred_col_store.get( model_name )

    if estimator is None or len( feature_cols ) == 0 or pred_col is None:
        continue

    X_train_model = phase6_train_model[ feature_cols ].fillna( fill_values )

    if len( phase6_holdout_model ) > 0:
        X_holdout_model = phase6_holdout_model[ feature_cols ].fillna( fill_values )
        phase6_holdout_model[ pred_col ] = estimator.predict( X_holdout_model )
        append_phase6_score( phase6_scores_rows, model_name, y_holdout, phase6_holdout_model[ pred_col ] )

    oof_pred = build_phase6_group_oof_predictions( 
        estimator,
        X_train_model,
        y_train,
        phase6_train_model[ "station_group" ],
    )

    calibrator = LinearRegression( )
    calibrator.fit( np.asarray( oof_pred ).reshape( -1, 1 ), y_train )
    calibrated_model = Phase6CalibratedPredictor( estimator, calibrator )
    calibrated_name = f"{ model_name }_calibrated"
    calibrated_pred_col = f"{ pred_col }_calibrated"

    phase6_model_store[ calibrated_name ] = calibrated_model
    phase6_feature_store[ calibrated_name ] = feature_cols
    phase6_fill_store[ calibrated_name ] = fill_values
    phase6_pred_col_store[ calibrated_name ] = calibrated_pred_col

    oof_calibrated = calibrator.predict( np.asarray( oof_pred ).reshape( -1, 1 ) )
    phase6_calibration_rows.append( 
        { 
            "model": model_name,
            "calibration_type": "linear",
            "slope": float( calibrator.coef_[ 0 ] ),
            "intercept": float( calibrator.intercept_ ),
            "oof_bias_raw": float( np.mean( oof_pred - y_train ) ),
            "oof_bias_calibrated": float( np.mean( oof_calibrated - y_train ) ),
            "oof_mae_raw": float( mean_absolute_error( y_train, oof_pred ) ),
            "oof_mae_calibrated": float( mean_absolute_error( y_train, oof_calibrated ) ),
        }
    )

    if len( phase6_holdout_model ) > 0:
        phase6_holdout_model[ calibrated_pred_col ] = calibrated_model.predict( X_holdout_model )
        append_phase6_score( phase6_scores_rows, calibrated_name, y_holdout, phase6_holdout_model[ calibrated_pred_col ] )

    isotonic_calibrator = IsotonicRegression( out_of_bounds = "clip" )
    isotonic_calibrator.fit( np.asarray( oof_pred, dtype = "float64" ), np.asarray( y_train, dtype = "float64" ) )
    isotonic_model = Phase6CalibratedPredictor( estimator, isotonic_calibrator )
    isotonic_name = f"{ model_name }_isotonic"
    isotonic_pred_col = f"{ pred_col }_isotonic"

    phase6_model_store[ isotonic_name ] = isotonic_model
    phase6_feature_store[ isotonic_name ] = feature_cols
    phase6_fill_store[ isotonic_name ] = fill_values
    phase6_pred_col_store[ isotonic_name ] = isotonic_pred_col

    oof_isotonic = isotonic_calibrator.predict( np.asarray( oof_pred, dtype = "float64" ) )
    phase6_calibration_rows.append( 
        { 
            "model": model_name,
            "calibration_type": "isotonic",
            "slope": np.nan,
            "intercept": np.nan,
            "oof_bias_raw": float( np.mean( oof_pred - y_train ) ),
            "oof_bias_calibrated": float( np.mean( oof_isotonic - y_train ) ),
            "oof_mae_raw": float( mean_absolute_error( y_train, oof_pred ) ),
            "oof_mae_calibrated": float( mean_absolute_error( y_train, oof_isotonic ) ),
        }
    )

    if len( phase6_holdout_model ) > 0:
        phase6_holdout_model[ isotonic_pred_col ] = isotonic_model.predict( X_holdout_model )
        append_phase6_score( phase6_scores_rows, isotonic_name, y_holdout, phase6_holdout_model[ isotonic_pred_col ] )


### 6.3 Phase 6 Results

This cell shows the operational leaderboard, the calibration summary, and the selected learned model.


In [ ]:
phase6_scores = pd.DataFrame( phase6_scores_rows )
phase6_scores_operational = phase6_scores.sort_values( "rmse", ascending = True ).reset_index( drop = True )
phase6_scores_scenario = phase6_scores.loc[ 
    phase6_scores[ "model" ].str.startswith( ( "ridge", "hgb" ) )
].sort_values( "rmse", ascending = True ).reset_index( drop = True )
phase6_calibration_summary = pd.DataFrame( phase6_calibration_rows ).round( 4 )

# for demonstration, we'll select the best air-only ridge model as the "operational" model 
# to carry forward into phase 7 ...
phase6_selected_name = str( phase6_scores_scenario.iloc[ 0 ][ "model" ] ) if len( phase6_scores_scenario ) > 0 else "ridge_air_only"
phase6_selected_model = phase6_model_store[ phase6_selected_name ]
phase6_selected_features = phase6_feature_store[ phase6_selected_name ]
phase6_selected_fill_values = phase6_fill_store[ phase6_selected_name ]
phase6_selected_pred_col = phase6_pred_col_store[ phase6_selected_name ]

print( "phase 6 leaderboard ( operational ):" )
display( phase6_scores_operational.round( 4 ) )


In [ ]:
print( "phase 6 calibration summary:" )
display( phase6_calibration_summary )


In [ ]:
print( f"selected phase6 model: { phase6_selected_name }" )
print( f"selected feature count: { len( phase6_selected_features ) }" )

### 6.4 Warming Sanity Check

This is a small directional test: warm one row by 2 C in the air features and confirm the selected model responds in the expected direction.


In [ ]:
# and now, let's do one crude sanity check by warming a single row
if len( phase6_holdout_model ) > 0:
    phase6_demo_ready = phase6_holdout_model.copy( )
    phase6_demo_ready[ phase6_selected_features ] = phase6_demo_ready[ phase6_selected_features ].fillna( phase6_selected_fill_values )
    phase6_demo_row = phase6_demo_ready[ [ "region", "station", "date", target_col ] + phase6_selected_features ].head( 1 )

    if len( phase6_demo_row ) > 0:
        phase6_demo_warm = phase6_demo_row.copy( )
        warm_cols = [ col for col in phase6_selected_features if col.startswith( "air_temp" ) ]
        for col in warm_cols:
            phase6_demo_warm[ col ] = phase6_demo_warm[ col ] + 2.0

        baseline_pred = float( phase6_selected_model.predict( phase6_demo_row[ phase6_selected_features ] )[ 0 ] )
        warm_pred = float( phase6_selected_model.predict( phase6_demo_warm[ phase6_selected_features ] )[ 0 ] )

        print( "phase 6 +2 C sanity check:" )
        print( phase6_demo_row[ [ "region", "station", "date", target_col ] ] )
        print( f"baseline predicted delta_water_temp: { baseline_pred:.3f}" )
        print( f"+2 C predicted delta_water_temp: { warm_pred:.3f}" )
        print( f"implied delta shift from warming: { warm_pred - baseline_pred:.3f}" )


## Phase 7: Water Response Models

Intent:
- use the phase 6 predicted warming signal plus station context to model salinity, oxygen, and pH responses
- keep one best learned model per target for later forecasting

Result to expect:
- `phase7_targets`
- `phase7_model_store`
- `phase7_feature_store`
- `phase7_fill_store`


### 7.0 Build Station-Year Response Tables

This cell prepares the annual and warm-season station-year tables that feed the phase-7 water-property models.


In [ ]:
# and now, let's prep the station-year response table
air_source_7 = add_phase6_engineered_features( daily_air_final.merge( properties_baseline, on = [ "region", "station" ], how = "left" ) )
water_source_7 = daily_water_final

phase7_base_target_candidates = [ "delta_salinity", "delta_oxygen", "delta_ph" ]
phase7_base_targets = [ col for col in phase7_base_target_candidates if col in water_source_7.columns ]

water_keep_cols_7 = [ "region", "station", "date" ] + phase7_base_targets
for col in [ "oxygen", "water_temp_baseline", "salinity_baseline", "oxygen_baseline", "ph_baseline", "depth_baseline" ]:
    if col in water_source_7.columns:
        water_keep_cols_7.append( col )
water_keep_cols_7 = list( dict.fromkeys( water_keep_cols_7 ) )

phase7_daily = air_source_7.merge( 
    water_source_7[ water_keep_cols_7 ],
    on = [ "region", "station", "date" ],
    how = "inner",
)
phase7_daily = phase7_daily.merge( phase6_context_table, on = [ "region", "station" ], how = "left" )

phase7_cluster_lookup = station_baseline[ [ "region", "station", "cluster" ] ]
phase7_cluster_lookup[ "cluster_code" ] = pd.to_numeric( phase7_cluster_lookup[ "cluster" ], errors = "coerce" )
phase7_cluster_lookup = phase7_cluster_lookup.drop( columns = [ "cluster" ] )
phase7_daily = phase7_daily.merge( phase7_cluster_lookup, on = [ "region", "station" ], how = "left" )
phase7_daily = add_phase6_engineered_features( phase7_daily )

phase7_p6_feature_cols = list( phase6_selected_features )
for col in phase7_p6_feature_cols:
    if col not in phase7_daily.columns:
        phase7_daily[ col ] = np.nan

phase7_p6_fill_values = phase6_selected_fill_values.reindex( phase7_p6_feature_cols )
X_phase7_for_p6 = phase7_daily[ phase7_p6_feature_cols ].fillna( phase7_p6_fill_values )
phase7_daily[ "delta_water_temp_pred_p6" ] = phase6_selected_model.predict( X_phase7_for_p6 )

phase7_daily[ "year" ] = pd.to_datetime( phase7_daily[ "date" ], errors = "coerce" ).dt.year
phase7_daily[ "month_num" ] = pd.to_datetime( phase7_daily[ "date" ], errors = "coerce" ).dt.month
phase7_daily[ "is_warm_season" ] = phase7_daily[ "month_num" ].between( 6, 9 )

agg_feature_cols = [ col for col in [ "delta_water_temp_pred_p6", "air_temp", "precip", "wind_speed", "solar" ] if col in phase7_daily.columns ]
baseline_daily_feature_cols = [ col for col in [ "water_temp_baseline", "salinity_baseline", "oxygen_baseline", "ph_baseline", "depth_baseline" ] if col in phase7_daily.columns ]

phase7_annual_agg = { "n_days_annual": ( "date", "size" ) }
for col in agg_feature_cols:
    phase7_annual_agg[ f"{ col }_mean" ] = ( col, "mean" )
    phase7_annual_agg[ f"{ col }_std" ] = ( col, "std" )
for col in baseline_daily_feature_cols:
    phase7_annual_agg[ f"{ col }_annual_mean" ] = ( col, "mean" )
for target in phase7_base_targets:
    phase7_annual_agg[ f"{ target }_annual_mean" ] = ( target, "mean" )

phase7_station_year = ( 
    phase7_daily
    .groupby( [ "region", "station", "year" ], as_index = False )
    .agg( **phase7_annual_agg )
)

phase7_warm_daily = phase7_daily.loc[ phase7_daily[ "is_warm_season" ] ]
phase7_warm_agg = { "n_days_warm": ( "date", "size" ) }
for col in agg_feature_cols:
    phase7_warm_agg[ f"{ col }_warm_mean" ] = ( col, "mean" )
    phase7_warm_agg[ f"{ col }_warm_std" ] = ( col, "std" )
for target in phase7_base_targets:
    phase7_warm_agg[ f"{ target }_warm_mean" ] = ( target, "mean" )

phase7_warm_station_year = ( 
    phase7_warm_daily
    .groupby( [ "region", "station", "year" ], as_index = False )
    .agg( **phase7_warm_agg )
)
phase7_station_year = phase7_station_year.merge( phase7_warm_station_year, on = [ "region", "station", "year" ], how = "left" )


def warm_quantile( values, q ):
    valid = pd.Series( values ).dropna( )
    return float( valid.quantile( q ) ) if len( valid ) > 0 else np.nan


def warm_share_below( values, threshold ):
    valid = pd.Series( values ).dropna( )
    return float( ( valid < threshold ).mean( ) ) if len( valid ) > 0 else np.nan


if "oxygen" in phase7_warm_daily.columns:
    phase7_oxygen_stress = ( 
        phase7_warm_daily
        .groupby( [ "region", "station", "year" ], as_index = False )
        .agg( 
            oxygen_warm_min = ( "oxygen", "min" ),
            oxygen_warm_p10 = ( "oxygen", lambda values: warm_quantile( values, 0.10 ) ),
            oxygen_warm_hypoxic_day_share = ( "oxygen", lambda values: warm_share_below( values, 2.0 ) ),
            oxygen_warm_below_optimal_day_share = ( "oxygen", lambda values: warm_share_below( values, 5.0 ) ),
        )
    )
    phase7_station_year = phase7_station_year.merge( phase7_oxygen_stress, on = [ "region", "station", "year" ], how = "left" )

phase7_station_year = phase7_station_year.merge( 
    station_year_quality[ 
        [ 
            "region",
            "station",
            "year",
            "has_model_year",
            "has_warm_oxygen_year",
            "n_temp_days",
            "n_temp_months",
            "n_warm_oxygen_days",
            "n_warm_oxygen_months",
        ]
    ],
    on = [ "region", "station", "year" ],
    how = "left",
)
phase7_station_year[ "has_model_year" ] = phase7_station_year[ "has_model_year" ].fillna( False ).astype( bool )
phase7_station_year[ "has_warm_oxygen_year" ] = phase7_station_year[ "has_warm_oxygen_year" ].fillna( False ).astype( bool )


### 7.0 Response Targets And Feature Sets

This cell defines the station context, target columns, feature sets, and target-specific coverage rules.


In [ ]:
phase7_station_context_feature_candidates = [ 
    "mean_annual_water_temp",
    "seasonal_amp_water_temp",
    "mean_annual_salinity",
    "seasonal_amp_salinity",
    "mean_annual_oxygen",
    "seasonal_amp_oxygen",
    "mean_annual_saturation",
    "seasonal_amp_saturation",
    "mean_annual_ph",
    "seasonal_amp_ph",
    "mean_annual_depth",
    "seasonal_amp_depth",
    "warm_peak_month",
    "oxygen_nadir_month",
    "temp_oxygen_phase_lag",
    "warm_len_months",
    "cold_len_months",
    "warm_intensity_temp",
    "cold_intensity_temp",
    "warm_oxygen_mean",
    "warm_oxygen_min",
    "cold_oxygen_mean",
    "n_obs_total",
    "n_years_present",
    "mean_year_coverage",
    "is_partial_baseline",
]
phase7_station_context_cols = [ col for col in phase7_station_context_feature_candidates if col in station_baseline.columns ]
phase7_station_context = station_baseline[ [ "region", "station" ] + phase7_station_context_cols ]
phase7_station_context = phase7_station_context.merge( phase7_cluster_lookup, on = [ "region", "station" ], how = "left" )
phase7_station_year = phase7_station_year.merge( phase7_station_context, on = [ "region", "station" ], how = "left" )

phase7_station_year_train = phase7_station_year.merge( unflagged_station_keys_final, on = [ "region", "station" ], how = "inner" )
phase7_station_year_holdout = phase7_station_year.merge( flagged_station_keys_final, on = [ "region", "station" ], how = "inner" )

phase7_feature_candidates_static = [ 
    "water_temp_baseline_annual_mean",
    "salinity_baseline_annual_mean",
    "oxygen_baseline_annual_mean",
    "ph_baseline_annual_mean",
    "depth_baseline_annual_mean",
    "cluster_code",
    "mean_annual_water_temp",
    "seasonal_amp_water_temp",
    "mean_annual_salinity",
    "seasonal_amp_salinity",
    "mean_annual_oxygen",
    "seasonal_amp_oxygen",
    "mean_annual_saturation",
    "seasonal_amp_saturation",
    "mean_annual_ph",
    "seasonal_amp_ph",
    "mean_annual_depth",
    "seasonal_amp_depth",
    "warm_peak_month",
    "oxygen_nadir_month",
    "temp_oxygen_phase_lag",
    "warm_len_months",
    "cold_len_months",
    "warm_intensity_temp",
    "cold_intensity_temp",
    "warm_oxygen_mean",
    "warm_oxygen_min",
    "cold_oxygen_mean",
    "n_obs_total",
    "n_years_present",
    "mean_year_coverage",
    "is_partial_baseline",
]
phase7_feature_candidates_annual = [ 
    "delta_water_temp_pred_p6_mean",
    "air_temp_mean",
    "precip_mean",
    "wind_speed_mean",
    "solar_mean",
    "delta_water_temp_pred_p6_std",
    "air_temp_std",
    "precip_std",
    "wind_speed_std",
    "solar_std",
] + phase7_feature_candidates_static
phase7_feature_candidates_warm = [ 
    "delta_water_temp_pred_p6_warm_mean",
    "air_temp_warm_mean",
    "precip_warm_mean",
    "wind_speed_warm_mean",
    "solar_warm_mean",
    "delta_water_temp_pred_p6_warm_std",
    "air_temp_warm_std",
    "precip_warm_std",
    "wind_speed_warm_std",
    "solar_warm_std",
] + phase7_feature_candidates_static

phase7_features_annual = [ col for col in dict.fromkeys( phase7_feature_candidates_annual ) if col in phase7_station_year_train.columns ]
phase7_features_warm = [ col for col in dict.fromkeys( phase7_feature_candidates_warm ) if col in phase7_station_year_train.columns ]
phase7_use_warm_oxygen_features = len( phase7_features_warm ) > 0

phase7_target_col_map = { 
    "delta_salinity": "delta_salinity_annual_mean" if "delta_salinity_annual_mean" in phase7_station_year_train.columns else None,
    "delta_oxygen": "delta_oxygen_warm_mean" if "delta_oxygen_warm_mean" in phase7_station_year_train.columns else ( "delta_oxygen_annual_mean" if "delta_oxygen_annual_mean" in phase7_station_year_train.columns else None ),
    "oxygen_warm_min": "oxygen_warm_min" if "oxygen_warm_min" in phase7_station_year_train.columns else None,
    "oxygen_warm_p10": "oxygen_warm_p10" if "oxygen_warm_p10" in phase7_station_year_train.columns else None,
    "oxygen_warm_hypoxic_day_share": "oxygen_warm_hypoxic_day_share" if "oxygen_warm_hypoxic_day_share" in phase7_station_year_train.columns else None,
    "oxygen_warm_below_optimal_day_share": "oxygen_warm_below_optimal_day_share" if "oxygen_warm_below_optimal_day_share" in phase7_station_year_train.columns else None,
    "delta_ph": "delta_ph_annual_mean" if "delta_ph_annual_mean" in phase7_station_year_train.columns else None,
}
phase7_target_feature_map = { 
    "delta_salinity": list( phase7_features_annual ),
    "delta_oxygen": list( phase7_features_warm if phase7_use_warm_oxygen_features else phase7_features_annual ),
    "oxygen_warm_min": list( phase7_features_warm if phase7_use_warm_oxygen_features else phase7_features_annual ),
    "oxygen_warm_p10": list( phase7_features_warm if phase7_use_warm_oxygen_features else phase7_features_annual ),
    "oxygen_warm_hypoxic_day_share": list( phase7_features_warm if phase7_use_warm_oxygen_features else phase7_features_annual ),
    "oxygen_warm_below_optimal_day_share": list( phase7_features_warm if phase7_use_warm_oxygen_features else phase7_features_annual ),
    "delta_ph": list( phase7_features_annual ),
}
phase7_target_year_flag_map = { 
    "delta_salinity": "has_model_year",
    "delta_oxygen": "has_warm_oxygen_year" if phase7_use_warm_oxygen_features else "has_model_year",
    "oxygen_warm_min": "has_warm_oxygen_year" if phase7_use_warm_oxygen_features else "has_model_year",
    "oxygen_warm_p10": "has_warm_oxygen_year" if phase7_use_warm_oxygen_features else "has_model_year",
    "oxygen_warm_hypoxic_day_share": "has_warm_oxygen_year" if phase7_use_warm_oxygen_features else "has_model_year",
    "oxygen_warm_below_optimal_day_share": "has_warm_oxygen_year" if phase7_use_warm_oxygen_features else "has_model_year",
    "delta_ph": "has_model_year",
}
phase7_target_window_label_map = { 
    "delta_salinity": "annual mean",
    "delta_oxygen": "warm-season mean" if phase7_target_col_map.get( "delta_oxygen" ) == "delta_oxygen_warm_mean" else "annual mean",
    "oxygen_warm_min": "warm-season minimum",
    "oxygen_warm_p10": "warm-season 10th percentile",
    "oxygen_warm_hypoxic_day_share": "warm-season share < 2 mg/L",
    "oxygen_warm_below_optimal_day_share": "warm-season share < 5 mg/L",
    "delta_ph": "annual mean",
}
phase7_share_targets = [ "oxygen_warm_hypoxic_day_share", "oxygen_warm_below_optimal_day_share" ]
phase7_targets = [ 
    target
    for target in [ 
        "delta_salinity",
        "delta_oxygen",
        "oxygen_warm_min",
        "oxygen_warm_p10",
        "oxygen_warm_hypoxic_day_share",
        "oxygen_warm_below_optimal_day_share",
        "delta_ph",
    ]
    if phase7_target_col_map.get( target ) is not None and len( phase7_target_feature_map.get( target, [ ] ) ) > 0
]


### 7.1 Phase 7 Helper Classes

These helpers keep the response-model fitting cell focused on the training loop rather than repeated plumbing.


In [ ]:
class Phase7TargetPredictor:


    def __init__( self, base_model, target_name, calibrator = None ):
        self.base_model = base_model
        self.target_name = target_name
        self.calibrator = calibrator


    def predict( self, X ):
        pred = np.asarray( self.base_model.predict( X ), dtype = "float64" )

        if self.calibrator is not None:
            pred = self.calibrator.predict( pred.reshape( -1, 1 ) )

        if self.target_name in phase7_share_targets:
            pred = np.clip( pred, 0.0, 1.0 )

        return pred


def append_phase7_score( score_rows, target, target_window, model_name, y_true, y_pred, n_train, n_holdout, n_features, coverage_flag, target_col ):
    score_rows.append( 
        { 
            "target": target,
            "target_window": target_window,
            "model": model_name,
            "mae": float( mean_absolute_error( y_true, y_pred ) ),
            "rmse": float( mean_squared_error( y_true, y_pred ) ** 0.5 ),
            "r2": float( r2_score( y_true, y_pred ) ),
            "bias": float( np.mean( np.asarray( y_pred ) - np.asarray( y_true ) ) ),
            "n_train": int( n_train ),
            "n_holdout": int( n_holdout ),
            "n_features": int( n_features ),
            "coverage_flag": coverage_flag,
            "target_col": target_col,
        }
    )


def build_phase7_sample_weights( y, target ):
    y = pd.Series( y, copy = False ).astype( "float64" )
    weights = pd.Series( 1.0, index = y.index, dtype = "float64" )
    valid = y.dropna( )

    if len( valid ) < 8:
        return weights

    if target in [ "delta_oxygen", "oxygen_warm_min", "oxygen_warm_p10" ]:
        q25 = float( valid.quantile( 0.25 ) )
        q10 = float( valid.quantile( 0.10 ) )
        weights.loc[ y <= q25 ] = 2.0
        weights.loc[ y <= q10 ] = 3.0

    elif target in phase7_share_targets:
        q75 = float( valid.quantile( 0.75 ) )
        q90 = float( valid.quantile( 0.90 ) )
        weights.loc[ y >= q75 ] = 2.0
        weights.loc[ y >= q90 ] = 3.0

    else:
        q20 = float( valid.quantile( 0.20 ) )
        q80 = float( valid.quantile( 0.80 ) )
        weights.loc[ ( y <= q20 ) | ( y >= q80 ) ] = 1.75

    return weights


def build_phase7_group_oof_predictions( estimator, X, y, groups, sample_weight = None ):
    groups = pd.Series( groups, index = X.index ).astype( str )
    sample_weight = pd.Series( sample_weight, index = X.index ).astype( "float64" ) if sample_weight is not None else None
    unique_groups = groups.dropna( ).unique( )

    if len( unique_groups ) < 2:
        fitted_estimator = clone( estimator )
        if sample_weight is None:
            fitted_estimator.fit( X, y )

        else:
            fitted_estimator.fit( X, y, sample_weight = sample_weight )
        return pd.Series( fitted_estimator.predict( X ), index = X.index, dtype = "float64" )

    splitter = GroupKFold( n_splits = min( 5, len( unique_groups ) ) )
    oof_pred = pd.Series( index = X.index, dtype = "float64" )

    for fit_idx, valid_idx in splitter.split( X, y, groups ):
        fold_model = clone( estimator )
        if sample_weight is None:
            fold_model.fit( X.iloc[ fit_idx ], y.iloc[ fit_idx ] )

        else:
            fold_model.fit( X.iloc[ fit_idx ], y.iloc[ fit_idx ], sample_weight = sample_weight.iloc[ fit_idx ] )
        oof_pred.iloc[ valid_idx ] = fold_model.predict( X.iloc[ valid_idx ] )

    if oof_pred.isna( ).any( ):
        fallback_model = clone( estimator )
        if sample_weight is None:
            fallback_model.fit( X, y )

        else:
            fallback_model.fit( X, y, sample_weight = sample_weight )
        missing_mask = oof_pred.isna( )
        oof_pred.loc[ missing_mask ] = fallback_model.predict( X.loc[ missing_mask ] )

    return oof_pred


### 7.1 Fit Response Models

This cell trains the candidate models for each response target and stores the best scenario model for later forecasting.


In [ ]:
phase7_scores_rows = [ ]
phase7_best_rows = [ ]
phase7_model_store = { }
phase7_feature_store = { }
phase7_fill_store = { }

for target in phase7_targets:
    target_col = phase7_target_col_map.get( target )
    feature_cols_t = phase7_target_feature_map.get( target, [ ] )
    year_flag_col = phase7_target_year_flag_map.get( target, "has_model_year" )

    train_t = phase7_station_year_train.loc[ phase7_station_year_train[ year_flag_col ].fillna( False ).astype( bool ) ]
    holdout_t = phase7_station_year_holdout.loc[ phase7_station_year_holdout[ year_flag_col ].fillna( False ).astype( bool ) ]
    train_t = train_t.dropna( subset = [ target_col ] )
    holdout_t = holdout_t.dropna( subset = [ target_col ] )

    if len( train_t ) == 0 or len( holdout_t ) == 0:
        continue

    y_train_t = train_t[ target_col ]
    y_holdout_t = holdout_t[ target_col ]
    X_train_t = train_t[ feature_cols_t ].apply( pd.to_numeric, errors = "coerce" )
    X_holdout_t = holdout_t[ feature_cols_t ].apply( pd.to_numeric, errors = "coerce" )

    usable_feature_cols_t = X_train_t.columns[ X_train_t.notna( ).any( axis = 0 ) ].tolist( )
    if len( usable_feature_cols_t ) == 0:
        continue

    X_train_t = X_train_t[ usable_feature_cols_t ]
    X_holdout_t = X_holdout_t.reindex( columns = usable_feature_cols_t )
    fill_values_t = X_train_t.median( numeric_only = True ).reindex( usable_feature_cols_t ).fillna( 0.0 )
    X_train_t = X_train_t.fillna( fill_values_t )
    X_holdout_t = X_holdout_t.fillna( fill_values_t )
    feature_cols_t = usable_feature_cols_t
    station_groups_t = train_t[ "region" ].astype( str ) + " | " + train_t[ "station" ].astype( str )
    sample_weight_t = build_phase7_sample_weights( y_train_t, target )

    pred_naive = np.full( len( y_holdout_t ), float( y_train_t.median( ) ) )
    pred_naive = np.clip( pred_naive, 0.0, 1.0 ) if target in phase7_share_targets else pred_naive
    append_phase7_score( 
        phase7_scores_rows,
        target,
        phase7_target_window_label_map.get( target, "annual mean" ),
        "naive_global_median",
        y_holdout_t,
        pred_naive,
        len( train_t ),
        len( holdout_t ),
        len( feature_cols_t ),
        year_flag_col,
        target_col,
    )

    model_candidates_t = { }

    ridge_t = Ridge( alpha = 1.0 )
    ridge_t.fit( X_train_t, y_train_t )
    model_candidates_t[ "ridge_phase7_climate" ] = Phase7TargetPredictor( ridge_t, target )

    ridge_weighted_t = Ridge( alpha = 1.0 )
    ridge_weighted_t.fit( X_train_t, y_train_t, sample_weight = sample_weight_t )
    ridge_oof_t = build_phase7_group_oof_predictions( Ridge( alpha = 1.0 ), X_train_t, y_train_t, station_groups_t, sample_weight = sample_weight_t )
    ridge_calibrator_t = LinearRegression( )
    ridge_calibrator_t.fit( np.asarray( ridge_oof_t ).reshape( -1, 1 ), y_train_t, sample_weight = sample_weight_t )
    model_candidates_t[ "ridge_phase7_weighted_calibrated" ] = Phase7TargetPredictor( ridge_weighted_t, target, ridge_calibrator_t )

    hgb_t = HistGradientBoostingRegressor( 
        learning_rate = 0.05,
        max_depth = 4,
        max_iter = 300,
        min_samples_leaf = 20,
        random_state = 42,
    )
    hgb_t.fit( X_train_t, y_train_t )
    model_candidates_t[ "hgb_phase7_climate" ] = Phase7TargetPredictor( hgb_t, target )

    hgb_weighted_t = HistGradientBoostingRegressor( 
        learning_rate = 0.05,
        max_depth = 4,
        max_iter = 300,
        min_samples_leaf = 20,
        random_state = 42,
    )
    hgb_weighted_t.fit( X_train_t, y_train_t, sample_weight = sample_weight_t )
    hgb_oof_t = build_phase7_group_oof_predictions( 
        HistGradientBoostingRegressor( 
            learning_rate = 0.05,
            max_depth = 4,
            max_iter = 300,
            min_samples_leaf = 20,
            random_state = 42,
        ),
        X_train_t,
        y_train_t,
        station_groups_t,
        sample_weight = sample_weight_t,
    )
    hgb_calibrator_t = LinearRegression( )
    hgb_calibrator_t.fit( np.asarray( hgb_oof_t ).reshape( -1, 1 ), y_train_t, sample_weight = sample_weight_t )
    model_candidates_t[ "hgb_phase7_weighted_calibrated" ] = Phase7TargetPredictor( hgb_weighted_t, target, hgb_calibrator_t )

    for model_name, model_obj in model_candidates_t.items( ):
        pred_t = model_obj.predict( X_holdout_t )
        append_phase7_score( 
            phase7_scores_rows,
            target,
            phase7_target_window_label_map.get( target, "annual mean" ),
            model_name,
            y_holdout_t,
            pred_t,
            len( train_t ),
            len( holdout_t ),
            len( feature_cols_t ),
            year_flag_col,
            target_col,
        )

    target_scores = pd.DataFrame( phase7_scores_rows )
    target_scores = target_scores.loc[ target_scores[ "target" ] == target ].sort_values( "rmse", ascending = True )
    target_scores_non_naive = target_scores.loc[ target_scores[ "model" ] != "naive_global_median" ]
    best_row_for_scenarios = target_scores_non_naive.iloc[ 0 ] if len( target_scores_non_naive ) > 0 else target_scores.iloc[ 0 ]

    best_model_name = str( best_row_for_scenarios[ "model" ] )
    phase7_model_store[ target ] = model_candidates_t[ best_model_name ]
    phase7_feature_store[ target ] = feature_cols_t
    phase7_fill_store[ target ] = fill_values_t
    phase7_best_rows.append( 
        { 
            "target": target,
            "target_window": phase7_target_window_label_map.get( target, "annual mean" ),
            "best_model_for_scenarios": best_model_name,
            "best_rmse_for_scenarios": float( best_row_for_scenarios[ "rmse" ] ),
            "best_mae_for_scenarios": float( best_row_for_scenarios[ "mae" ] ),
            "best_r2_for_scenarios": float( best_row_for_scenarios[ "r2" ] ),
            "best_bias_for_scenarios": float( best_row_for_scenarios[ "bias" ] ),
            "n_train": int( len( train_t ) ),
            "n_holdout": int( len( holdout_t ) ),
            "n_features": int( len( feature_cols_t ) ),
            "coverage_flag": year_flag_col,
            "target_col": target_col,
        }
    )


### 7.1 Phase 7 Results

This cell prints the per-target holdout leaderboard and the best selected model for each target.


In [ ]:
phase7_scores = pd.DataFrame( phase7_scores_rows ).sort_values( [ "target", "rmse" ], ascending = [ True, True ] ).reset_index( drop = True )
phase7_best = pd.DataFrame( phase7_best_rows ).sort_values( "best_rmse_for_scenarios" ).reset_index( drop = True )

print( "phase7 response leaderboard ( holdout ):" )
display( phase7_scores.round( 4 ) )


In [ ]:
print( "phase7 best model per target:" )
display( phase7_best.round( 4 ) )

## Phase 9: Forward Projection

Intent:
- build simple scenario paths
- apply the phase 6 and phase 7 models to future daily and station-year tables
- summarize projected regime shifts and hypoxia risks

Result to expect:
- case-study forecast tables
- a portfolio-style station scan
- a saved model bundle under `estuaries/artifacts`


### 9.0 Forward-Projection Helpers

These helpers define how case studies are selected and how future daily and station-year driver tables are built.


In [ ]:
# 9.0 - forward-projection helpers + case-study setup
from helpers.model_bundle import build_t4d_model_bundle, save_t4d_model_bundle
from helpers.phase9_scan import run_phase9_station_scan

phase9_case_count = 5
phase9_case_pool_mode = "holdout_transition"
phase9_case_region_codes = [ "cbm", "cbv" ]
phase9_case_station_keys = None
phase9_rolling_window_years = 5
phase9_scenario_path_csv = ref_dir / "phase9.scenario.paths.csv"


def select_phase9_case_studies( case_pool, n_cases = 5 ):
    case_pool = case_pool.sort_values( [ "event_date", "region", "station" ] ).reset_index( drop = True )
    selected_rows = [ ]
    used_keys = set( )

    for cluster_id in case_pool[ "cluster" ].dropna( ).astype( int ).sort_values( ).unique( ):
        cluster_slice = case_pool.loc[ case_pool[ "cluster" ].astype( "Int64" ) == int( cluster_id ) ]
        if len( cluster_slice ) == 0:
            continue

        row = cluster_slice.iloc[ 0 ]
        key = ( str( row[ "region" ] ), str( row[ "station" ] ) )
        if key in used_keys:
            continue

        selected_rows.append( row )
        used_keys.add( key )
        if len( selected_rows ) >= n_cases:
            break

    if len( selected_rows ) < n_cases:
        for row in case_pool.itertuples( index = False ):
            key = ( str( row.region ), str( row.station ) )
            if key in used_keys:
                continue

            selected_rows.append( pd.Series( row._asdict( ) ) )
            used_keys.add( key )
            if len( selected_rows ) >= n_cases:
                break

    out = pd.DataFrame( selected_rows ).reset_index( drop = True )
    out[ "case_id" ] = [ f"case_{ idx + 1 }" for idx in range( len( out ) ) ]
    return out


def normalize_phase9_station_keys( station_keys ):
    if station_keys is None:
        return None

    rows = [ ]
    for idx, item in enumerate( station_keys ):
        if isinstance( item, dict ):
            region = item.get( "region" )
            station = item.get( "station" )

        elif isinstance( item, ( list, tuple ) ) and len( item ) >= 2:
            region, station = item[ 0 ], item[ 1 ]

        else:
            continue

        if pd.isna( region ) or pd.isna( station ):
            continue

        rows.append( 
            { 
                "region": str( region ).strip( ),
                "station": str( station ).strip( ),
                "phase9_case_order": idx,
            }
        )

    return pd.DataFrame( rows ).drop_duplicates( subset = [ "region", "station" ] ).reset_index( drop = True ) if rows else None


def build_phase9_case_pool( case_pool_mode, case_station_keys = None ):
    station_meta = station_baseline_display[ 
        [ "region", "station", "region_name", "station_name", "cluster", "cluster_label", "cluster_name" ]
    ].drop_duplicates( )
    available_keys = ( 
        daily_air_final[ [ "region", "station" ] ]
        .drop_duplicates( )
        .merge( daily_water_final[ [ "region", "station" ] ].drop_duplicates( ), on = [ "region", "station" ], how = "inner" )
    )
    transition_pool = ( 
        first_event
        .merge( station_meta, on = [ "region", "station" ], how = "left" )
        .merge( flagged_station_keys_final.assign( is_holdout = True ), on = [ "region", "station" ], how = "left" )
        .merge( available_keys.assign( has_history = True ), on = [ "region", "station" ], how = "inner" )
    )
    transition_pool[ "is_holdout" ] = transition_pool[ "is_holdout" ].fillna( False )
    transition_pool[ "is_train" ] = ~transition_pool[ "is_holdout" ]

    if case_pool_mode == "holdout_transition":
        case_pool = transition_pool.loc[ transition_pool[ "is_holdout" ] ]

    elif case_pool_mode == "train_transition":
        case_pool = transition_pool.loc[ transition_pool[ "is_train" ] ]

    elif case_pool_mode == "all_transition":
        case_pool = transition_pool

    elif case_pool_mode == "station_list":
        station_keys_frame = normalize_phase9_station_keys( case_station_keys )
        if station_keys_frame is None or len( station_keys_frame ) == 0:
            raise ValueError( "phase9_case_station_keys must be provided when phase9_case_pool_mode = 'station_list'" )

        case_pool = ( 
            station_keys_frame
            .merge( station_meta, on = [ "region", "station" ], how = "left" )
            .merge( first_event, on = [ "region", "station" ], how = "left" )
            .merge( flagged_station_keys_final.assign( is_holdout = True ), on = [ "region", "station" ], how = "left" )
            .merge( available_keys.assign( has_history = True ), on = [ "region", "station" ], how = "inner" )
            .sort_values( [ "phase9_case_order", "region", "station" ] )
            .reset_index( drop = True )
        )
        case_pool[ "is_holdout" ] = case_pool[ "is_holdout" ].fillna( False )
        case_pool[ "is_train" ] = ~case_pool[ "is_holdout" ]

    else:
        raise ValueError( f"Unknown phase9_case_pool_mode: { case_pool_mode }" )

    return case_pool.sort_values( [ "event_date", "region", "station" ], na_position = "last" ).reset_index( drop = True )


def build_phase9_demo_paths( start_year = 2026, end_year = 2100 ):
    rows = [ ]
    demo_specs = [ 
        ( "ssp245_demo", 1.8, 1.00, 1.00, 1.00 ),
        ( "ssp585_demo", 3.7, 0.97, 1.02, 1.00 ),
    ]
    n_years = max( 1, end_year - start_year )

    for scenario_name, end_air_add_c, end_precip_mult, end_wind_mult, end_solar_mult in demo_specs:
        for year in range( start_year, end_year + 1 ):
            frac = ( year - start_year ) / n_years
            rows.append( 
                { 
                    "scenario": scenario_name,
                    "year": year,
                    "air_temp_add_c": float( end_air_add_c * frac ),
                    "precip_mult": float( 1.0 + ( end_precip_mult - 1.0 ) * frac ),
                    "wind_speed_mult": float( 1.0 + ( end_wind_mult - 1.0 ) * frac ),
                    "solar_mult": float( 1.0 + ( end_solar_mult - 1.0 ) * frac ),
                }
            )

    return pd.DataFrame( rows )


def add_phase9_driver_rolls( frame ):
    frame = frame.sort_values( [ "region", "station", "scenario", "date" ] )

    for source_col in [ "air_temp", "wind_speed", "precip", "solar" ]:
        for window in [ 1, 7, 28 ]:
            frame[ f"{ source_col }_r{ window }d" ] = ( 
                frame
                .groupby( [ "region", "station", "scenario" ] )[ source_col ]
                .transform( lambda values: values.shift( 1 ).rolling( window = window, min_periods = 1 ).mean( ) )
            )

    frame[ "doy" ] = frame[ "date" ].dt.dayofyear
    frame[ "doy_sin" ] = np.sin( 2 * np.pi * frame[ "doy" ] / 365.25 )
    frame[ "doy_cos" ] = np.cos( 2 * np.pi * frame[ "doy" ] / 365.25 )
    return frame


def build_phase9_station_year_features( phase9_daily_frame ):
    frame = phase9_daily_frame.dropna( subset = [ "date" ] )
    frame[ "date" ] = pd.to_datetime( frame[ "date" ], errors = "coerce" )
    frame[ "year" ] = frame[ "date" ].dt.year
    frame[ "month_num" ] = frame[ "date" ].dt.month
    frame[ "is_warm_season" ] = frame[ "month_num" ].between( 6, 9 )

    agg_feature_cols = [ col for col in [ "delta_water_temp_pred_p6", "air_temp", "precip", "wind_speed", "solar" ] if col in frame.columns ]
    baseline_daily_feature_cols = [ col for col in [ "water_temp_baseline", "salinity_baseline", "oxygen_baseline", "ph_baseline", "depth_baseline" ] if col in frame.columns ]

    annual_agg = { "n_days_annual": ( "date", "size" ) }
    for col in agg_feature_cols:
        annual_agg[ f"{ col }_mean" ] = ( col, "mean" )
        annual_agg[ f"{ col }_std" ] = ( col, "std" )
    for col in baseline_daily_feature_cols:
        annual_agg[ f"{ col }_annual_mean" ] = ( col, "mean" )

    station_year = ( 
        frame
        .groupby( [ "region", "station", "scenario", "year" ], as_index = False )
        .agg( **annual_agg )
    )

    warm_daily = frame.loc[ frame[ "is_warm_season" ] ]
    warm_agg = { "n_days_warm": ( "date", "size" ) }
    for col in agg_feature_cols:
        warm_agg[ f"{ col }_warm_mean" ] = ( col, "mean" )
        warm_agg[ f"{ col }_warm_std" ] = ( col, "std" )

    warm_year = ( 
        warm_daily
        .groupby( [ "region", "station", "scenario", "year" ], as_index = False )
        .agg( **warm_agg )
    )
    station_year = station_year.merge( warm_year, on = [ "region", "station", "scenario", "year" ], how = "left" )
    station_year = station_year.merge( phase7_station_context, on = [ "region", "station" ], how = "left" )
    station_year[ "has_model_year" ] = True
    station_year[ "has_warm_oxygen_year" ] = True
    return station_year


### 9.0 Regime Classification Helper

This helper maps rolling projected conditions back into the learned baseline regime space.


In [ ]:
phase9_regime_feature_cols = list( centroids_z.columns )


def phase9_classify_regime_row( row, regime_feature_cols = None, min_features_required = 3 ):
    regime_feature_cols = phase9_regime_feature_cols if regime_feature_cols is None else regime_feature_cols
    available = [ 
        col
        for col in regime_feature_cols
        if ( 
            col in row.index
            and col in feature_mean.index
            and col in feature_scale.index
            and col in centroids_z.columns
            and pd.notna( row.get( col, np.nan ) )
        )
    ]

    if len( available ) < min_features_required:
        return pd.Series( 
            { 
                "regime_roll5y": np.nan,
                "regime_dist_best": np.nan,
                "regime_margin": np.nan,
                "regime_n_features_used": len( available ),
            }
        )

    values = pd.to_numeric( pd.Series( { col: row.get( col, np.nan ) for col in available } ), errors = "coerce" )
    means = pd.to_numeric( feature_mean[ available ], errors = "coerce" )
    scales = pd.to_numeric( feature_scale[ available ], errors = "coerce" ).replace( 0, np.nan )
    z_values = ( values - means ) / scales

    if z_values.isna( ).any( ):
        return pd.Series( 
            { 
                "regime_roll5y": np.nan,
                "regime_dist_best": np.nan,
                "regime_margin": np.nan,
                "regime_n_features_used": len( available ),
            }
        )

    centroid_slice = centroids_z[ available ]
    centroid_array = np.asarray( centroid_slice.to_numpy( ), dtype = "float64" )
    z_array = np.asarray( z_values.to_numpy( ), dtype = "float64" )
    distances = np.sqrt( np.sum( ( centroid_array - z_array[ None, : ] ) ** 2, axis = 1 ) )

    best_idx = int( np.argmin( distances ) )
    best_cluster = int( centroid_slice.index[ best_idx ] )
    best_dist = float( distances[ best_idx ] )
    margin = float( np.partition( distances, 1 )[ 1 ] - best_dist ) if len( distances ) > 1 else np.nan

    return pd.Series( 
        { 
            "regime_roll5y": best_cluster,
            "regime_dist_best": best_dist,
            "regime_margin": margin,
            "regime_n_features_used": len( available ),
        }
    )


### 9.1 Build Case Studies And Historical Anchors

This cell picks the stations to project, loads the scenario paths, and builds the historical driver climatology.


In [ ]:
# 9.1 - build case-study forecasts from annual scenario paths
phase9_case_pool = build_phase9_case_pool( phase9_case_pool_mode, case_station_keys = phase9_case_station_keys )
if phase9_case_region_codes:
    phase9_case_region_codes_norm = { str( code ).strip( ).lower( ) for code in phase9_case_region_codes }
    phase9_case_pool = phase9_case_pool.loc[ phase9_case_pool[ "region" ].astype( str ).str.lower( ).isin( phase9_case_region_codes_norm ) ]

phase9_case_studies = select_phase9_case_studies( phase9_case_pool, n_cases = phase9_case_count )
phase9_case_keys = phase9_case_studies[ [ "region", "station" ] ].drop_duplicates( )

if phase9_scenario_path_csv.exists( ):
    phase9_scenario_paths = pd.read_csv( phase9_scenario_path_csv )
    phase9_path_source = f"external_csv: { phase9_scenario_path_csv }"

else:
    phase9_scenario_paths = build_phase9_demo_paths( start_year = 2026, end_year = 2100 )
    phase9_path_source = "demo_ramps"

phase9_scenario_paths[ "scenario" ] = phase9_scenario_paths[ "scenario" ].astype( str )
phase9_scenario_paths[ "year" ] = pd.to_numeric( phase9_scenario_paths[ "year" ], errors = "coerce" ).astype( "Int64" )
for col, fill_value in [ 
    ( "air_temp_add_c", 0.0 ),
    ( "precip_mult", 1.0 ),
    ( "wind_speed_mult", 1.0 ),
    ( "solar_mult", 1.0 ),
]:
    if col not in phase9_scenario_paths.columns:
        phase9_scenario_paths[ col ] = fill_value

    phase9_scenario_paths[ col ] = pd.to_numeric( phase9_scenario_paths[ col ], errors = "coerce" ).fillna( fill_value )

phase9_scenarios = sorted( phase9_scenario_paths[ "scenario" ].dropna( ).unique( ).tolist( ) )
phase9_future_year_min = int( phase9_scenario_paths[ "year" ].dropna( ).min( ) )
phase9_future_year_max = int( phase9_scenario_paths[ "year" ].dropna( ).max( ) )

phase9_hist_air = daily_air_final.merge( phase9_case_keys, on = [ "region", "station" ], how = "inner" )
phase9_hist_water = daily_water_final.merge( phase9_case_keys, on = [ "region", "station" ], how = "inner" )
phase9_hist_air[ "date" ] = pd.to_datetime( phase9_hist_air[ "date" ], errors = "coerce" )
phase9_hist_water[ "date" ] = pd.to_datetime( phase9_hist_water[ "date" ], errors = "coerce" )

phase9_hist_join = phase9_hist_air.merge( 
    phase9_hist_water[ [ "region", "station", "date", "water_temp", "salinity", "oxygen", "depth" ] ],
    on = [ "region", "station", "date" ],
    how = "inner",
)
phase9_hist_join[ "year" ] = phase9_hist_join[ "date" ].dt.year
phase9_hist_join[ "month_num" ] = phase9_hist_join[ "date" ].dt.month
phase9_hist_join[ "is_warm_season" ] = phase9_hist_join[ "month_num" ].between( 6, 9 )

phase9_hist_annual = ( 
    phase9_hist_join
    .groupby( [ "region", "station", "year" ], as_index = False )
    .agg( 
        air_temp_mean = ( "air_temp", "mean" ),
        precip_mean = ( "precip", "mean" ),
        water_temp_abs_annual_mean = ( "water_temp", "mean" ),
        salinity_abs_annual_mean = ( "salinity", "mean" ),
        oxygen_abs_annual_mean = ( "oxygen", "mean" ),
        depth_abs_annual_mean = ( "depth", "mean" ),
    )
)
phase9_hist_warm = ( 
    phase9_hist_join.loc[ phase9_hist_join[ "is_warm_season" ] ]
    .groupby( [ "region", "station", "year" ], as_index = False )
    .agg( oxygen_warm_min_abs = ( "oxygen", "min" ) )
)
phase9_hist_annual = phase9_hist_annual.merge( phase9_hist_warm, on = [ "region", "station", "year" ], how = "left" )
phase9_hist_annual[ "scenario" ] = "observed_history"

phase9_properties_baseline = properties_baseline
phase9_case_meta = phase9_case_studies[ 
    [ "region", "station", "case_id", "region_name", "station_name", "cluster", "cluster_label", "cluster_name", "event_date" ]
]
phase9_hist_annual = phase9_hist_annual.merge( phase9_case_meta, on = [ "region", "station" ], how = "left" )
phase9_hist_annual = phase9_hist_annual.merge( phase9_properties_baseline, on = [ "region", "station" ], how = "left" )

phase9_climo = phase9_hist_air
phase9_climo[ "doy" ] = phase9_climo[ "date" ].dt.dayofyear.clip( upper = 365 )
phase9_driver_climatology = ( 
    phase9_climo
    .groupby( [ "region", "station", "doy" ], as_index = False )
    .agg( 
        air_temp = ( "air_temp", "mean" ),
        precip = ( "precip", "mean" ),
        wind_speed = ( "wind_speed", "mean" ),
        solar = ( "solar", "mean" ),
    )
)


### 9.1 Build Future Forecast Tables

This cell constructs the future daily driver table, applies phase 6 and phase 7, and rolls the results into station-year forecast summaries.


In [ ]:
phase9_driver_station_mean = ( 
    phase9_hist_air
    .groupby( [ "region", "station" ], as_index = False )
    .agg( 
        air_temp_station_mean = ( "air_temp", "mean" ),
        precip_station_mean = ( "precip", "mean" ),
        wind_speed_station_mean = ( "wind_speed", "mean" ),
        solar_station_mean = ( "solar", "mean" ),
    )
)

phase9_future_dates = pd.DataFrame( { "date": pd.date_range( f"{ phase9_future_year_min }-01-01", f"{ phase9_future_year_max }-12-31", freq = "D" ) } )
phase9_future_dates[ "year" ] = phase9_future_dates[ "date" ].dt.year
phase9_future_dates[ "doy" ] = phase9_future_dates[ "date" ].dt.dayofyear.clip( upper = 365 )
phase9_future_dates = phase9_future_dates.loc[ phase9_future_dates[ "year" ].isin( phase9_scenario_paths[ "year" ].dropna( ).astype( int ) ) ]

phase9_future_daily = ( 
    phase9_case_keys.assign( _tmp = 1 )
    .merge( pd.DataFrame( { "scenario": phase9_scenarios, "_tmp": 1 } ), on = "_tmp", how = "inner" )
    .merge( phase9_future_dates.assign( _tmp = 1 ), on = "_tmp", how = "inner" )
    .drop( columns = [ "_tmp" ] )
)
phase9_future_daily = phase9_future_daily.merge( phase9_driver_climatology, on = [ "region", "station", "doy" ], how = "left" )
phase9_future_daily = phase9_future_daily.merge( phase9_driver_station_mean, on = [ "region", "station" ], how = "left" )
phase9_future_daily = phase9_future_daily.merge( phase9_scenario_paths, on = [ "scenario", "year" ], how = "left" )

for col, station_fill_col in [ 
    ( "air_temp", "air_temp_station_mean" ),
    ( "precip", "precip_station_mean" ),
    ( "wind_speed", "wind_speed_station_mean" ),
    ( "solar", "solar_station_mean" ),
]:
    phase9_future_daily[ col ] = phase9_future_daily[ col ].fillna( phase9_future_daily[ station_fill_col ] )

phase9_future_daily[ "air_temp" ] = phase9_future_daily[ "air_temp" ] + phase9_future_daily[ "air_temp_add_c" ]
phase9_future_daily[ "precip" ] = phase9_future_daily[ "precip" ] * phase9_future_daily[ "precip_mult" ]
phase9_future_daily[ "wind_speed" ] = phase9_future_daily[ "wind_speed" ] * phase9_future_daily[ "wind_speed_mult" ]
phase9_future_daily[ "solar" ] = phase9_future_daily[ "solar" ] * phase9_future_daily[ "solar_mult" ]
phase9_future_daily = add_phase9_driver_rolls( phase9_future_daily )

phase9_future_daily = phase9_future_daily.merge( phase9_properties_baseline, on = [ "region", "station" ], how = "left" )
phase9_future_daily = phase9_future_daily.merge( phase6_context_table, on = [ "region", "station" ], how = "left" )
phase9_cluster_lookup = phase7_cluster_lookup
phase9_future_daily = phase9_future_daily.merge( phase9_cluster_lookup, on = [ "region", "station" ], how = "left" )
phase9_future_daily = add_phase6_engineered_features( phase9_future_daily )

for col in phase6_selected_features:
    if col not in phase9_future_daily.columns:
        phase9_future_daily[ col ] = np.nan

phase9_p6_X = phase9_future_daily[ phase6_selected_features ].fillna( phase6_selected_fill_values.reindex( phase6_selected_features ) )
phase9_future_daily[ "delta_water_temp_pred_p6" ] = phase6_selected_model.predict( phase9_p6_X )
phase9_future_daily[ "water_temp_pred" ] = phase9_future_daily[ "water_temp_baseline" ] + phase9_future_daily[ "delta_water_temp_pred_p6" ]

phase9_future_station_year = build_phase9_station_year_features( phase9_future_daily )
for target in phase7_targets:
    model_t = phase7_model_store.get( target )
    feature_cols_t = phase7_feature_store.get( target, [ ] )
    fill_values_t = phase7_fill_store.get( target, pd.Series( dtype = "float64" ) )
    if model_t is None or len( feature_cols_t ) == 0:
        continue

    X_target_t = phase9_future_station_year[ feature_cols_t ].fillna( fill_values_t.reindex( feature_cols_t ) )
    phase9_future_station_year[ f"{ target }_pred" ] = model_t.predict( X_target_t )

phase9_future_station_year[ "water_temp_abs_annual_mean" ] = phase9_future_station_year[ "water_temp_baseline_annual_mean" ] + phase9_future_station_year[ "delta_water_temp_pred_p6_mean" ]
phase9_future_station_year[ "salinity_abs_annual_mean" ] = phase9_future_station_year[ "salinity_baseline_annual_mean" ] + phase9_future_station_year.get( "delta_salinity_pred", 0.0 )
phase9_future_station_year[ "oxygen_abs_annual_mean" ] = phase9_future_station_year[ "oxygen_baseline_annual_mean" ] + phase9_future_station_year.get( "delta_oxygen_pred", np.nan )
phase9_future_station_year[ "oxygen_warm_min_abs" ] = phase9_future_station_year.get( "oxygen_warm_min_pred", np.nan )
phase9_future_station_year[ "oxygen_plot_abs" ] = phase9_future_station_year[ "oxygen_warm_min_abs" ]
phase9_future_station_year.loc[ phase9_future_station_year[ "oxygen_plot_abs" ].isna( ), "oxygen_plot_abs" ] = phase9_future_station_year.loc[ phase9_future_station_year[ "oxygen_plot_abs" ].isna( ), "oxygen_abs_annual_mean" ]
phase9_future_station_year = phase9_future_station_year.merge( phase9_case_meta, on = [ "region", "station" ], how = "left" )

phase9_history_plot = phase9_hist_annual
phase9_history_plot[ "oxygen_plot_abs" ] = phase9_history_plot[ "oxygen_warm_min_abs" ]
phase9_history_plot.loc[ phase9_history_plot[ "oxygen_plot_abs" ].isna( ), "oxygen_plot_abs" ] = phase9_history_plot.loc[ phase9_history_plot[ "oxygen_plot_abs" ].isna( ), "oxygen_abs_annual_mean" ]

phase9_metric_roll_map = { 
    "air_temp_mean": "air_temp_roll5y",
    "precip_mean": "precip_roll5y",
    "water_temp_abs_annual_mean": "water_temp_roll5y",
    "salinity_abs_annual_mean": "salinity_roll5y",
    "oxygen_abs_annual_mean": "oxygen_roll5y",
    "oxygen_plot_abs": "oxygen_plot_roll5y",
}
for source_col, out_col in phase9_metric_roll_map.items( ):
    phase9_history_plot[ out_col ] = ( 
        phase9_history_plot
        .sort_values( [ "region", "station", "year" ] )
        .groupby( [ "region", "station" ] )[ source_col ]
        .transform( lambda values: values.rolling( window = phase9_rolling_window_years, min_periods = 3 ).mean( ) )
    )

phase9_history_plot[ "mean_annual_water_temp" ] = phase9_history_plot[ "water_temp_roll5y" ]
phase9_history_plot[ "mean_annual_salinity" ] = phase9_history_plot[ "salinity_roll5y" ]
phase9_history_plot[ "mean_annual_oxygen" ] = phase9_history_plot[ "oxygen_roll5y" ]
phase9_history_plot[ "mean_annual_depth" ] = phase9_history_plot[ "depth_baseline" ]
phase9_history_plot = pd.concat( [ phase9_history_plot, phase9_history_plot.apply( phase9_classify_regime_row, axis = 1 ) ], axis = 1 )

phase9_projection_paths = [ ]
for scenario_name in phase9_scenarios:
    hist_path = phase9_hist_annual.copy( )
    hist_path[ "scenario" ] = scenario_name
    future_path = phase9_future_station_year.loc[ phase9_future_station_year[ "scenario" ] == scenario_name ]
    path_frame = pd.concat( [ hist_path, future_path ], ignore_index = True, sort = False )
    path_frame = path_frame.sort_values( [ "region", "station", "year" ] ).reset_index( drop = True )

    for source_col, out_col in phase9_metric_roll_map.items( ):
        path_frame[ out_col ] = ( 
            path_frame
            .groupby( [ "region", "station" ] )[ source_col ]
            .transform( lambda values: values.rolling( window = phase9_rolling_window_years, min_periods = 3 ).mean( ) )
        )

    path_frame[ "mean_annual_water_temp" ] = path_frame[ "water_temp_roll5y" ]
    path_frame[ "mean_annual_salinity" ] = path_frame[ "salinity_roll5y" ]
    path_frame[ "mean_annual_oxygen" ] = path_frame[ "oxygen_roll5y" ]
    path_frame[ "mean_annual_depth" ] = path_frame[ "depth_baseline" ]
    path_frame = pd.concat( [ path_frame, path_frame.apply( phase9_classify_regime_row, axis = 1 ) ], axis = 1 )
    phase9_projection_paths.append( path_frame )

phase9_projection_plot = pd.concat( phase9_projection_paths, ignore_index = True ) if phase9_projection_paths else pd.DataFrame( )
phase9_projection_future_only = phase9_projection_plot.loc[ phase9_projection_plot[ "year" ] >= phase9_future_year_min ]

phase9_crossing_summary = ( 
    phase9_projection_future_only
    .loc[ 
        phase9_projection_future_only[ "regime_roll5y" ].notna( )
        & phase9_projection_future_only[ "cluster" ].notna( )
        & ( phase9_projection_future_only[ "regime_roll5y" ].astype( "Int64" ) != phase9_projection_future_only[ "cluster" ].astype( "Int64" ) )
    ]
    .groupby( [ "region", "station", "scenario" ], as_index = False )[ "year" ]
    .min( )
    .rename( columns = { "year": "first_regime_cross_year" } )
)
phase9_crossing_summary = phase9_case_meta.merge( phase9_crossing_summary, on = [ "region", "station" ], how = "left" )

phase9_signature_2100 = phase9_future_station_year.loc[ 
    phase9_future_station_year[ "year" ] == phase9_future_year_max,
    [ "region", "station", "scenario", "water_temp_abs_annual_mean", "salinity_abs_annual_mean", "oxygen_abs_annual_mean", "oxygen_warm_min_abs" ],
]
phase9_signature_2100 = phase9_case_meta.merge( phase9_signature_2100, on = [ "region", "station" ], how = "left" )


### 9.1 Forward-Projection Results

This cell prints the main case-study outputs from the forecast path.


In [ ]:
print( f"phase9 path source: { phase9_path_source }" )
print( f"phase9 case-study stations: { len( phase9_case_studies ) }" )
print( f"phase9 scenarios: { phase9_scenarios }" )
print( f"phase9 future station-year rows: { len( phase9_future_station_year ) }" )

print( "phase9 selected case studies:" )
display( phase9_case_studies[ [ "case_id", "region", "station", "station_name", "region_name", "cluster_label", "event_date" ] ] )
print( "phase9 regime crossing summary:" )
display( phase9_crossing_summary[ [ "case_id", "region", "station", "station_name", "scenario", "cluster_label", "first_regime_cross_year" ] ] )
print( "phase9 2100 signature snapshot:" )
display( phase9_signature_2100.round( 3 ) )


### 9.3 Run The Station Scan

This cell runs the broader station scan for projected regime crossings and hypoxia risk.


In [ ]:
# 9.3 - scan all eligible stations for projected regime crossings + hypoxia risk
phase9_scan_region_codes = None
phase9_scan_top_n = 12
phase9_scan_plot_n = 6
phase9_scan_hypoxia_threshold = 2.0
phase9_scan_material_share = 0.05
phase9_scan_progress_every = 20

phase9_scan_result = run_phase9_station_scan( 
    station_baseline_display = station_baseline_display,
    daily_air_final = daily_air_final,
    daily_water_final = daily_water_final,
    first_event = first_event,
    flagged_station_keys_final = flagged_station_keys_final,
    phase9_scenario_path_csv = phase9_scenario_path_csv,
    build_phase9_demo_paths = build_phase9_demo_paths,
    add_phase9_driver_rolls = add_phase9_driver_rolls,
    build_phase9_station_year_features = build_phase9_station_year_features,
    phase9_classify_regime_row = phase9_classify_regime_row,
    phase9_rolling_window_years = phase9_rolling_window_years,
    phase6_context_table = phase6_context_table,
    phase6_selected_features = phase6_selected_features,
    phase6_selected_fill_values = phase6_selected_fill_values,
    phase6_selected_model = phase6_selected_model,
    phase7_targets = phase7_targets,
    phase7_feature_store = phase7_feature_store,
    phase7_fill_store = phase7_fill_store,
    phase7_model_store = phase7_model_store,
    station_baseline = station_baseline,
    properties_baseline = properties_baseline,
    phase9_scenario_paths = phase9_scenario_paths,
    phase9_scenarios = phase9_scenarios,
    phase9_future_year_min = phase9_future_year_min,
    phase9_future_year_max = phase9_future_year_max,
    phase9_future_dates = phase9_future_dates,
    phase9_properties_baseline = phase9_properties_baseline,
    phase9_cluster_lookup = phase9_cluster_lookup,
    region_codes = phase9_scan_region_codes,
    top_n = phase9_scan_top_n,
    plot_n = phase9_scan_plot_n,
    hypoxia_threshold = phase9_scan_hypoxia_threshold,
    material_share = phase9_scan_material_share,
    progress_every = phase9_scan_progress_every,
)


### 9.3 Scan Outputs

This cell prints the main scan summaries and the top projected stations.


In [ ]:
print( f"phase9 scan path source: { phase9_scan_result[ 'path_source' ] }" )
print( f"phase9 scan stations projected: { len( phase9_scan_result[ 'station_keys' ] ) }" )
print( f"phase9 scan scenarios: { phase9_scan_result[ 'scenarios' ] }" )
print( f"phase9 scan future station-year rows: { len( phase9_scan_result[ 'future_station_year' ] ) }" )
print( "phase9 earliest projected regime-crossing candidates:" )
display( phase9_scan_result[ "crossing_display" ] )
print( "phase9 strongest projected hypoxia candidates:" )
display( phase9_scan_result[ "hypoxia_display" ].round( 3 ) )


### 9.4 Save The Model Bundle

This cell writes the trained bundle to the artifacts directory so the learned objects can be reused outside the notebook.


In [ ]:
# 9.4 - save the trained model bundle
phase9_bundle_out_path = artifact_dir / "t4d.model.bundle.joblib"

phase9_model_bundle = build_t4d_model_bundle( 
    domain_feature_cols = domain_feature_cols,
    scaler_domain = scaler_domain,
    kmeans_model = kmeans_model,
    cluster_name_map = cluster_name_map,
    cluster_order = cluster_order,
    cluster_name_order = cluster_name_order,
    cluster_label_order = cluster_label_order,
    cluster_note_order = cluster_note_order,
    phase9_regime_feature_cols = phase9_regime_feature_cols,
    centroids_z = centroids_z,
    feature_mean = feature_mean,
    feature_scale = feature_scale,
    phase6_selected_model = phase6_selected_model,
    phase6_selected_features = phase6_selected_features,
    phase6_selected_fill_values = phase6_selected_fill_values,
    phase6_context_table = phase6_context_table,
    phase7_targets = phase7_targets,
    phase7_model_store = phase7_model_store,
    phase7_feature_store = phase7_feature_store,
    phase7_fill_store = phase7_fill_store,
    phase7_station_context = phase7_station_context,
)

phase9_bundle_info = save_t4d_model_bundle( phase9_model_bundle, phase9_bundle_out_path )
print( f"saved model bundle: { phase9_bundle_info[ 'path' ] }" )
print( f"bundle size: { phase9_bundle_info[ 'megabytes' ]:.2f} MB" )
print( f"phase6 features saved: { len( phase9_model_bundle[ 'phase6_air_to_water_temp' ][ 'selected_features' ] ) }" )
print( f"phase7 targets saved: { phase9_model_bundle[ 'phase7_signature_models' ][ 'targets' ] }" )


## Wrap Up

Follow-up intent:
- use this notebook as the compact end-to-end modeling and forecasting path
- keep experiments and presentation-only work in the larger notebook

Follow-up results to check after a run:
- the phase 6 leaderboard and selected model
- the phase 7 per-target leaderboard
- the phase 9 case-study and scan tables
- the saved bundle at `estuaries/artifacts/t4d.model.bundle.joblib`

If you want to tune behavior later, the first knobs worth touching are:
- station and station-year coverage thresholds
- phase 6 selected feature list
- phase 9 scenario paths CSV
